# 🔀 17 · Conditional probability, independence and Bayes' rule
### Conditioning restricts the sample space · joint tables and the law of total probability · the product and chain rules · independence and conditional independence · are neighbouring nucleotides independent? · P(·|B) is a probability measure · Bayes' rule · diagnostic tests and genetic risk · naïve Bayes for single cells

**BiologyToMath** · Part E — Probability · Module 17 of 28

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/math/tutorial/BiologyToMath/notebooks/17_conditional_probability_bayes.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, pandas, Matplotlib, `fractions.Fraction` (exact probabilities), SciPy (`scipy.special.gammaln`, `scipy.stats.poisson` — only to check our formulas) |
| **Prerequisites** | Module 16 (sample spaces, events, the axioms, inclusion–exclusion), Module 03 (sets, partitions, Venn diagrams), Module 04 (strings and codons). |
| **Estimated time** | 170 min (reading + running + the ✏️ exercises) |
| **02-680 topics covered** | Conditional probability and Bayes' rule: conditioning restricts the sample space, P(A|B) = P(A∩B)/P(B), P(A|Ω) = P(A); the two-coin examples; the flu/cough joint table; the sum rule (theorem of total probability) over a partition; the product rule and the chain rule for n events; independence; conditional independence (the regular/two-headed coin box, the A–B–C table, the die example) and why neither implies the other; independence of nucleotides in background sequence and in stop codons; factors that break independence (gene structure, regulatory motifs, linkage disequilibrium); P(·|B) satisfies the three axioms (card example); P(A|B) ≠ P(B|A); Bayes' rule — prior, likelihood, posterior, the expanded denominator; inverse probability reasoning; the flu example; genetic risk assessment (BRCA1); the naïve Bayes classifier (spam example; single-cell RNA-seq classification) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. Coin tosses, dice, people, DNA sequences, haplotypes and test results are **SIMULATED**, and the single-cell counts of Section 10 come from the toolkit's **SIMULATED** `simulate_cell_counts`. The flu/cough table, Bob's flu, the screening test, the BRCA1 figures, the spam probabilities and the haplotype frequencies are **illustrative numbers**, not clinical or population estimates. The three stop codons and a 52-card deck are typed in.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Compute** a conditional probability $P(A \mid B) = P(A \cap B)/P(B)$ from a sample space or a joint probability table, and **explain** it as a restriction of the sample space.
2. **Apply** the sum rule (law of total probability), the product rule and the chain rule, and **derive** the chain rule for $n$ events.
3. **Test** whether two events are independent, and **prove** that $P(A \cap B) = P(A)P(B) \iff P(A \mid B) = P(A)$ when $P(B) > 0$.
4. **Distinguish** independence from conditional independence with three worked counterexamples, and **verify** them exactly and by simulation.
5. **Decide** from sequence data whether neighbouring nucleotides are independent (background DNA, CpG depletion, stop codons, linkage disequilibrium).
6. **Prove** that $P(\cdot \mid B)$ satisfies the three axioms, and **show** with an example why $P(A \mid B) \ne P(B \mid A)$ and $P(A \mid B \cup C) \ne P(A \mid B) + P(A \mid C)$.
7. **Derive** Bayes' rule, **name** prior, likelihood, posterior and evidence, and **compute** posteriors for symptoms, screening tests and genetic risk — including with natural frequencies.
8. **Build** a naïve-Bayes classifier, from the spam example to a Poisson naïve Bayes that assigns single cells to cell types, using log-probabilities to avoid underflow.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports and course helpers | 5 min |
| 1 · Conditioning restricts the sample space | the definition; two coins | 12 min |
| 2 · Joint tables and the sum rule | flu and cough; the law of total probability; a mosaic plot | 15 min |
| 3 · Product and chain rules | a litter of knockout mice; a DNA string | 15 min |
| 4 · Independence | coins, dice, Mendel's second law; independent ≠ disjoint | 12 min |
| 5 · Conditional independence | the two-coin box; the A–B–C table; the die | 20 min |
| 6 · Are neighbouring nucleotides independent? | background DNA, CpG, stop codons, linkage disequilibrium | 18 min |
| 7 · $P(\cdot \mid B)$ is a probability measure | the axioms again; the card example; $P(A \mid B) \ne P(B \mid A)$ | 12 min |
| 8 · Bayes' rule | derivation; Bob's flu; posterior vs prior | 15 min |
| 9 · Diagnostic tests and genetic risk | base-rate fallacy, natural frequencies, BRCA1, a carrier sister | 18 min |
| 10 · Naïve Bayes | spam; classifying single cells with Poisson counts | 25 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 10 min |

## 🧫 The biological question

A patient walks into a clinic with a cough. How likely is it that she has the flu? A woman learns that she carries a
*BRCA1* mutation: how much does that change her risk of breast cancer? A single cell from a tumour biopsy contains 9
molecules of *CD3E* mRNA and none of *CD79A*: is it a T cell or a B cell? And why does the sequence "CG" appear so
rarely in human DNA, when "GC" is common?

All four questions have the same shape: **we learn something** (a symptom, a test result, a few counts, the previous
base) **and want to update a probability**. Module 16 gave probabilities with nothing known; this module adds the
single most useful idea in applied probability — **conditioning** — and its famous consequence, **Bayes' rule**.

| Biological question | Mathematical tool | Where |
|---|---|---|
| Given a cough, how likely is flu? | conditional probability; joint table; sum rule | §1, §2, §8 |
| What is the chance of a sequence of events (three knockout pups)? | product and chain rules | §3 |
| Do two genes assort independently? Are adjacent bases independent? | independence | §4, §6 |
| Why are two tosses of a coin of unknown type dependent? | conditional independence | §5 |
| What does a positive screening test mean? | Bayes' rule; sensitivity, specificity, prevalence | §8, §9 |
| How much does a *BRCA1* mutation change cancer risk? | prior → posterior | §9 |
| Which cell type is this cell? | naïve Bayes with Poisson counts | §10 |

> 🧪 **Simulated data, with a known truth.** Every data set here is simulated, so we know the true probabilities. Each
> rule is derived on paper, computed exactly with fractions, and then checked against a simulation of many people,
> sequences or cells.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import itertools
from fractions import Fraction
from scipy import stats
from scipy.special import gammaln

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py). `draw_venn` draws Venn diagrams of events, `simulate_dna` and `BACKGROUND_TRANSITIONS` generate DNA with and without dependence between neighbouring bases (Sections 3 and 6), and `simulate_cell_counts` with `CELL_TYPE_MARKERS` produces the single-cell counts of Section 10. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import itertools

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def _venn_order(x):
    """Sort key for Venn-diagram labels: numbers in numeric order first, then everything else as text."""
    if isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, bool):
        return (0, float(x), "")
    return (1, 0.0, str(x))


def draw_venn(sets: dict, universe=None, ax=None, show_elements: bool = True, max_items: int = 6,
              title: str | None = None, highlight=None):
    """Draw a Venn diagram of two or three sets with matplotlib only.

    ``sets`` maps a label to a Python set, e.g. ``{"S": {1, 2, 3}, "T": {3, 4}}``.  Each region
    shows its elements (or ``n = …`` if it has more than ``max_items``).  If ``universe`` is given,
    a rectangle is drawn and the elements outside every set are listed in a corner.
    ``highlight`` is a function of the membership tuple, e.g. ``lambda s, t: s and not t`` for
    S \\ T; the regions where it is true are shaded.  Returns the Axes.
    """
    labels = list(sets)
    if len(labels) not in (2, 3):
        raise ValueError("draw_venn draws two or three sets")
    if ax is None:
        _, ax = plt.subplots(figsize=(4.6, 3.8))
    if len(labels) == 2:
        centres = [(-0.55, 0.0), (0.55, 0.0)]
    else:
        centres = [(-0.55, 0.32), (0.55, 0.32), (0.0, -0.6)]
    radius = 1.0
    colours = ["tab:blue", "tab:orange", "tab:green"]
    # classify a fine grid of points to find each region (used for shading and label positions)
    gx, gy = np.meshgrid(np.linspace(-2.2, 2.2, 441), np.linspace(-2.0, 1.9, 391))
    inside = [((gx - cx) ** 2 + (gy - cy) ** 2) <= radius ** 2 for cx, cy in centres]
    if highlight is not None:
        mask = np.zeros_like(gx, dtype=bool)
        for member in itertools.product([True, False], repeat=len(labels)):
            if any(member) and highlight(*member):
                region = np.ones_like(gx, dtype=bool)
                for k, m in enumerate(member):
                    region &= inside[k] if m else ~inside[k]
                mask |= region
        ax.contourf(gx, gy, mask.astype(float), levels=[0.5, 1.5], colors=["#ffd54f"], alpha=0.8)
    for (cx, cy), lab, col in zip(centres, labels, colours):
        ax.add_patch(plt.Circle((cx, cy), radius, fill=False, lw=2, color=col))
        lx = cx + (-1.0 if cx < 0 else 1.0 if cx > 0 else 0.0) * 0.95
        ly = cy + (0.95 if cy >= 0 else -1.15)
        ax.text(lx, ly, lab, ha="center", va="center", fontsize=11, fontweight="bold", color=col)
    elements = set().union(*sets.values())
    for member in itertools.product([True, False], repeat=len(labels)):
        if not any(member):
            continue
        region_items = [x for x in elements if all((x in sets[l]) == m for l, m in zip(labels, member))]
        region = np.ones_like(gx, dtype=bool)
        for k, m in enumerate(member):
            region &= inside[k] if m else ~inside[k]
        if not region.any():
            continue
        px, py = gx[region].mean(), gy[region].mean()
        listing = ", ".join(str(x) for x in sorted(region_items, key=_venn_order))
        if show_elements and len(region_items) <= max_items and len(listing) <= 16:
            text = listing if region_items else "∅"
        else:
            text = f"n = {len(region_items)}"
        ax.text(px, py, text, ha="center", va="center", fontsize=8, wrap=True)
    if universe is not None:
        ax.add_patch(plt.Rectangle((-2.1, -1.9), 4.2, 3.7, fill=False, lw=1, color="0.4"))
        outside = sorted(set(universe) - elements, key=_venn_order)
        text = ", ".join(map(str, outside)) if len(outside) <= max_items else f"n = {len(outside)}"
        ax.text(-2.0, -1.8, "U: " + (text if outside else "∅"), fontsize=8, va="bottom", color="0.3")
    ax.set_xlim(-2.2, 2.2)
    ax.set_ylim(-2.0, 1.9)
    ax.set_aspect("equal")
    ax.axis("off")
    if title:
        ax.set_title(title)
    return ax


NUCLEOTIDES = ("A", "C", "G", "T")


BACKGROUND_TRANSITIONS = np.array([
    [0.30, 0.20, 0.29, 0.21],
    [0.32, 0.30, 0.08, 0.30],
    [0.25, 0.25, 0.29, 0.21],
    [0.18, 0.24, 0.29, 0.29],
])


def simulate_dna(length: int, transitions=None, initial=None, seed: int = 0) -> str:
    """Random DNA string.  Without ``transitions`` the bases are i.i.d. uniform; otherwise they
    follow a first-order Markov chain with the given 4 × 4 transition matrix (order A, C, G, T)
    and initial distribution ``initial`` (default uniform)."""
    rng = np.random.default_rng(seed)
    initial = np.full(4, 0.25) if initial is None else np.asarray(initial, dtype=float)
    if transitions is None:
        return "".join(rng.choice(list(NUCLEOTIDES), size=length, p=initial))
    P = np.asarray(transitions, dtype=float)
    states = [rng.choice(4, p=initial)]
    for _ in range(length - 1):
        states.append(rng.choice(4, p=P[states[-1]]))
    return "".join(NUCLEOTIDES[s] for s in states)


CELL_TYPE_MARKERS = {
    "T cell": ["CD3E", "CD3D", "IL7R"],
    "B cell": ["CD79A", "MS4A1", "CD19"],
    "macrophage": ["CD68", "LYZ", "CSF1R"],
    "fibroblast": ["COL1A1", "COL1A2", "DCN"],
}


def simulate_cell_counts(n_cells=(160, 120, 90, 70), cell_types=("T cell", "B cell", "macrophage", "fibroblast"),
                         n_background: int = 28, overdispersion: float = 0.0, seed: int = 680) -> dict:
    """SIMULATED single-cell RNA counts (molecules of each gene detected in each cell).

    Each cell type has a true mean count λ for every gene: its three ``CELL_TYPE_MARKERS`` genes are
    high (λ ≈ 6–15) in that type and low (≈ 0.1–0.5) elsewhere; ``n_background`` other genes share
    similar λ (≈ 0.2–4) in all types.  Counts are Poisson(λ); with ``overdispersion`` φ > 0 each
    cell's λ is first multiplied by a Gamma(1/φ, φ) factor (mean 1), which makes counts negative-
    binomial with variance λ + φλ² — closer to real data.

    Returns a dict with ``counts`` (cells × genes, int), ``labels`` (cell type of each cell),
    ``genes``, ``rates`` (DataFrame cell types × genes of true λ) and ``source``.
    """
    rng = np.random.default_rng(seed)
    markers = [g for t in cell_types for g in CELL_TYPE_MARKERS[t]]
    genes = markers + [f"bg{i:02d}" for i in range(1, n_background + 1)]
    rates = np.zeros((len(cell_types), len(genes)))
    bg = rng.uniform(0.2, 4.0, n_background)
    for t, ctype in enumerate(cell_types):
        for j, g in enumerate(markers):
            rates[t, j] = rng.uniform(6.0, 15.0) if g in CELL_TYPE_MARKERS[ctype] else rng.uniform(0.1, 0.5)
        rates[t, len(markers):] = bg * rng.uniform(0.8, 1.25, n_background)
    labels, counts = [], []
    for t, (ctype, n) in enumerate(zip(cell_types, n_cells)):
        lam = np.tile(rates[t], (n, 1))
        if overdispersion > 0:
            lam = lam * rng.gamma(1.0 / overdispersion, overdispersion, size=lam.shape)
        counts.append(rng.poisson(lam))
        labels += [ctype] * n
    return {"counts": np.vstack(counts), "labels": np.array(labels), "genes": genes,
            "rates": pd.DataFrame(rates, index=list(cell_types), columns=genes),
            "source": "SIMULATED single-cell counts (BiologyToMath simulate_cell_counts)"}

print("🧰 toolkit loaded:", ", ".join(sorted(['BACKGROUND_TRANSITIONS', 'CELL_TYPE_MARKERS', 'NUCLEOTIDES', '_venn_order', 'draw_venn', 'simulate_cell_counts', 'simulate_dna'])))

---
## 1 · Conditioning restricts the sample space

> 📍 **Where we are.** In Module 16 every probability was computed with nothing known about the outcome. Now we are
> told that some event $B$ has happened. Outcomes outside $B$ are no longer possible, so $B$ becomes our new sample
> space — that is all conditioning is.

**Definition.** For a probability space $(\Omega, \mathcal{A}, P)$ and events $A, B \in \mathcal{A}$ with $P(B) > 0$,
the **conditional probability of $A$ given $B$** is

$$P(A \mid B) = \frac{P(A \cap B)}{P(B)}.$$

*In words:* restrict attention to the outcomes in $B$ (the event we know happened); among them, $A$ still happens
exactly in $A \cap B$; divide by $P(B)$ so that the new, smaller sample space has total probability 1. The vertical bar
reads **"given"**. If we "condition" on the whole sample space nothing changes — using the axiom of total probability,

$$P(A \mid \Omega) = \frac{P(A \cap \Omega)}{P(\Omega)} = \frac{P(A)}{1} = P(A).$$

*In words:* knowing only that "something in $\Omega$ happened" is no information at all.

**Two coins.** Let $A_{\text{firstheads}} = \{HH, HT\}$ be the event that the first coin is heads and
$A_{\text{allheads}} = \{HH\}$ the event that both coins are heads. Because $A_{\text{allheads}} \subseteq
A_{\text{firstheads}}$, their intersection is $A_{\text{allheads}}$ itself, and

$$P(A_{\text{allheads}} \mid A_{\text{firstheads}}) = \frac{P(A_{\text{allheads}} \cap A_{\text{firstheads}})}{P(A_{\text{firstheads}})} = \frac{1/4}{1/2} = \frac12 .$$

*In words:* "the probability you end up with both coin flips being heads, given that the first coin was heads" is
1/2 — once the first coin is known, only the second coin is uncertain. The reverse question, "the probability the first
coin was heads given that both are heads", is $P(A_{\text{firstheads}} \mid A_{\text{allheads}}) = \frac{1/4}{1/4} = 1$:
if both are heads, the first is certainly heads.

In [ ]:
# ── 1.1 Pictures of conditioning ──
from matplotlib.patches import Ellipse

fig, axes = plt.subplots(1, 3, figsize=(11, 3.3))
ax = axes[0]
ax.add_patch(plt.Rectangle((0, 0), 2, 1.2, fill=False, lw=1.2))
ax.add_patch(plt.Circle((0.75, 0.6), 0.42, color="tab:green", alpha=0.45))
ax.add_patch(plt.Circle((1.25, 0.6), 0.42, color="tab:blue", alpha=0.45))
ax.text(0.5, 1.08, "A", color="tab:green", fontsize=12); ax.text(1.45, 1.08, "B", color="tab:blue", fontsize=12)
ax.text(1.0, 0.6, "A ∩ B", ha="center", va="center", fontsize=9); ax.text(1.88, 0.06, "Ω", fontsize=12)
ax.set_title("before: the whole sample space Ω")
ax = axes[1]
ax.add_patch(plt.Circle((1.0, 0.6), 0.55, color="tab:blue", alpha=0.25))
clip = plt.Circle((1.0, 0.6), 0.55, transform=ax.transData)
lens = plt.Circle((0.35, 0.6), 0.55, color="tab:green", alpha=0.6)
ax.add_patch(lens); lens.set_clip_path(clip)
ax.add_patch(plt.Circle((1.0, 0.6), 0.55, fill=False, lw=1.5, color="tab:blue"))
ax.text(0.62, 0.6, "A ∩ B", ha="center", va="center", fontsize=9)
ax.text(1.0, 1.22, "B is the new Ω", ha="center", color="tab:blue")
ax.set_title("after learning B: P(A | B) = P(A ∩ B) / P(B)")
ax = axes[2]
ax.add_patch(plt.Rectangle((0, 0), 2, 1.2, fill=False, lw=1.2))
ax.add_patch(Ellipse((1.0, 0.6), 1.6, 1.0, fill=False, lw=1.2))
ax.add_patch(Ellipse((0.75, 0.5), 0.55, 0.35, fill=False, lw=1.2))
ax.text(0.75, 0.5, "A_allheads", ha="center", va="center", fontsize=8)
ax.text(1.35, 0.75, "A_firstheads", ha="center", fontsize=8); ax.text(1.88, 1.05, "Ω", fontsize=12)
ax.set_title("A_allheads ⊆ A_firstheads ⊆ Ω")
for a in axes:
    a.set_xlim(-0.05, 2.05); a.set_ylim(-0.05, 1.35); a.set_aspect("equal"); a.axis("off")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Left: two overlapping events in $\Omega$. Middle: once we know that $B$ happened, everything
outside $B$ is discarded; $B$ is rescaled to be the whole world, and $A$ survives only as the piece $A \cap B$, so
$P(A \mid B)$ is the *share* of $B$ occupied by $A \cap B$. Right: the coin events are nested,
$A_{\text{allheads}} \subseteq A_{\text{firstheads}}$, which is why their intersection is simply $A_{\text{allheads}}$.
Now the numbers, exactly and by simulation.

In [ ]:
# ── 1.2 The two-coin conditional probabilities: exact and SIMULATED ──
Omega2 = {"".join(t) for t in itertools.product("HT", repeat=2)}
P2 = lambda event: Fraction(len(set(event) & Omega2), len(Omega2))      # uniform measure on {HH, HT, TH, TT}


def conditional(P, A: set, B: set) -> Fraction:
    """P(A | B) = P(A ∩ B) / P(B), defined only when P(B) > 0."""
    if P(B) == 0:
        raise ValueError("cannot condition on an event of probability 0")
    return P(A & B) / P(B)


A_first = {w for w in Omega2 if w[0] == "H"}
A_all = {"HH"}
print("P(all heads | first heads) =", P2(A_all & A_first), "/", P2(A_first), "=", conditional(P2, A_all, A_first))
print("P(first heads | all heads) =", P2(A_first & A_all), "/", P2(A_all), "=", conditional(P2, A_first, A_all))
print("P(all heads | Ω) =", conditional(P2, A_all, Omega2), "= P(all heads) =", P2(A_all))

rng = np.random.default_rng(SEED + 12)
tosses = rng.choice(["H", "T"], size=(100_000, 2))
first_h = tosses[:, 0] == "H"; both_h = (tosses == "H").all(axis=1)
print(f"\nSIMULATED: among {first_h.sum():,} pairs with first heads, fraction with both heads = {both_h[first_h].mean():.4f}")
print(f"SIMULATED: among {both_h.sum():,} pairs with both heads, fraction with first heads = {first_h[both_h].mean():.4f}")

🔍 **What just happened?** `conditional` implements the definition literally and refuses to divide by zero. The exact
answers are $\tfrac{1/4}{1/2} = \tfrac12$ and $\tfrac{1/4}{1/4} = 1$, and conditioning on $\Omega$ returns the
unconditional $P(A_{\text{allheads}}) = 1/4$. The simulation shows what conditioning means in data: we **keep only the
repetitions in which the condition holds** (`both_h[first_h]` selects the pairs with first heads) and compute the
fraction there. Among the roughly 50,000 pairs with a head first, about half have two heads; among the roughly 25,000
pairs with two heads, the first toss is heads every single time (fraction 1.0000). Order matters:
$P(A \mid B)$ and $P(B \mid A)$ are different numbers.

✏️ **Your turn.** (1) For three tosses compute $P(\text{at least two heads} \mid \text{first toss heads})$ exactly and by
simulation. (2) What happens in `conditional` if you condition on $A_{\text{threetails}} = \emptyset$? Why must the
definition exclude $P(B) = 0$?

---
## 2 · Joint probability tables and the sum rule

> 📍 **Where we are.** Real problems rarely come with a list of equally likely outcomes. They come with a **table** of
> probabilities for combinations of yes/no questions. Conditioning on such a table needs one more rule: how to get
> $P(B)$ from the table.

Consider the following (illustrative) scenario for people's illnesses: each person either has the flu ($F$) or not
($\overline{F}$), and either has a cough ($C$) or not ($\overline{C}$). The four combinations partition the population:

| Flu? | Cough? | $P(\text{flu status} \cap \text{cough status})$ |
|---|---|---|
| $F$ | $C$ | 0.30 |
| $F$ | $\overline{C}$ | 0.20 |
| $\overline{F}$ | $C$ | 0.15 |
| $\overline{F}$ | $\overline{C}$ | 0.35 |

How do we calculate the probability that a person has the flu given that she has a cough?
$P(F \mid C) = P(F \cap C)/P(C)$ — "the number of people with both a cough and the flu" divided by "the total number of
people with a cough". $P(C)$ is the chance that any single person, chosen at random from the whole group, has a cough:
$P(C) = (\text{number of people with a cough}) / (\text{number of people in the population})$. In the table, the people
with a cough are those with flu *and* cough plus those *without* flu and with cough.

In [ ]:
# ── 2.1 The flu/cough joint table: margins, P(C) and P(F | C) (ILLUSTRATIVE numbers) ──
def check_joint_table(table: pd.DataFrame) -> None:
    """A joint probability table must have non-negative entries that sum to exactly 1."""
    total = table.to_numpy().sum()
    if (table.to_numpy() < 0).any() or not np.isclose(total, 1.0):
        raise ValueError(f"not a joint distribution: entries sum to {total:.3f}")


joint = pd.DataFrame([[0.30, 0.20], [0.15, 0.35]], index=["F (flu)", "F̄ (no flu)"], columns=["C (cough)", "C̄ (no cough)"])
check_joint_table(joint)
with_margins = joint.copy()
with_margins["row total"] = joint.sum(axis=1)
with_margins.loc["column total"] = with_margins.sum(axis=0)
print(with_margins.round(3), "\n")
P_C = joint.loc["F (flu)", "C (cough)"] + joint.loc["F̄ (no flu)", "C (cough)"]
print(f"P(C) = P(C ∩ F) + P(C ∩ F̄) = 0.30 + 0.15 = {P_C:.2f}")
print(f"P(F | C) = P(F ∩ C) / P(C) = 0.30 / {P_C:.2f} = {0.30 / P_C:.4f}  (= 2/3)")
print(f"P(F) = {joint.loc['F (flu)'].sum():.2f}  — learning 'cough' raises the probability of flu from 0.50 to 0.67")
try:
    check_joint_table(pd.DataFrame([[0.30, 0.20], [0.15, 0.25]]))
except ValueError as err:
    print("\na mistyped table is caught:", err)

🔍 **What just happened?** The table's row totals give $P(F) = 0.5$ and $P(\overline{F}) = 0.5$; its column totals give
$P(C) = 0.45$ and $P(\overline{C}) = 0.55$ — these **margins** are where the name "marginal probability" comes from.
Adding the two cough cells gives $P(C) = 0.30 + 0.15 = 0.45$, and so $P(F \mid C) = 0.30/0.45 = 2/3 \approx 0.667$:
among coughing people, two thirds have the flu. ⚠️ Always check that a joint table sums to 1 — `check_joint_table`
caught a version with 0.25 in the last cell, whose entries sum to only 0.90 (no probability measure can produce it).

The step "add the cells of the cough column" is an instance of a general rule. Recall (Module 03) that a **partition**
of $\Omega$ is a collection of sets $A_1, \ldots, A_n$ that are pairwise disjoint and whose union is everything,
$\bigcup_{i=1}^n A_i = \Omega$.

**Sum rule.** If $A_1, \ldots, A_n$ partition $\Omega$, then for any event $B$

$$P(B) = \sum_{i=1}^{n} P(B \cap A_i).$$

*In words:* cut $B$ into the pieces that fall into each part of the partition; the pieces are disjoint and together
make up $B$, so by Axiom 3 their probabilities add up to $P(B)$. With the partition $\{F, \overline{F}\}$:
$P(C) = P(C \cap F) + P(C \cap \overline{F}) = 0.3 + 0.15$.

Writing each piece with the definition of conditional probability, $P(B \cap A_i) = P(B \mid A_i)\,P(A_i)$, gives the
**law of total probability** (the "theorem of total probability"):

$$P(B) = \sum_{i=1}^{n} P(B \mid A_i)\,P(A_i).$$

*In words:* the overall probability of $B$ is a weighted average of its probabilities within each group, weighted by
how big each group is. Here $P(C) = P(C \mid F)P(F) + P(C \mid \overline{F})P(\overline{F}) = 0.6 \times 0.5 + 0.3
\times 0.5 = 0.45$.

In [ ]:
# ── 2.2 The joint table as areas (a mosaic plot), and SIMULATED people ──
fig, ax = plt.subplots(figsize=(5.4, 3.6))
x0 = 0.0
for flu, colour in [("F (flu)", "tab:red"), ("F̄ (no flu)", "tab:blue")]:
    width = joint.loc[flu].sum()                                 # P(flu status): column width
    p_cough_given = joint.loc[flu, "C (cough)"] / width          # P(C | flu status): height of the cough block
    ax.add_patch(plt.Rectangle((x0, 0), width, p_cough_given, fc=colour, alpha=0.75, ec="k"))
    ax.add_patch(plt.Rectangle((x0, p_cough_given), width, 1 - p_cough_given, fc=colour, alpha=0.25, ec="k"))
    ax.text(x0 + width / 2, p_cough_given / 2, f"{flu[:2].strip()} ∩ C\n{joint.loc[flu, 'C (cough)']:.2f}", ha="center", va="center")
    ax.text(x0 + width / 2, (1 + p_cough_given) / 2, f"{flu[:2].strip()} ∩ C̄\n{joint.loc[flu, 'C̄ (no cough)']:.2f}", ha="center", va="center")
    ax.text(x0 + width / 2, -0.08, f"{flu}: {width:.2f}", ha="center", va="top", fontsize=8)
    x0 += width
ax.set_xlim(0, 1); ax.set_ylim(-0.15, 1); ax.set_yticks([0, 1]); ax.set_xticks([])
ax.set_ylabel("P(cough status | flu status)")
ax.set_title("area = joint probability; P(C) = 0.30 + 0.15 (dark blocks)")
plt.tight_layout(); plt.show()

rng = np.random.default_rng(SEED + 22)
cells = ["F∩C", "F∩C̄", "F̄∩C", "F̄∩C̄"]
people = rng.choice(cells, size=10_000, p=joint.to_numpy().ravel())
counts = pd.Series(people).value_counts().reindex(cells)
print("SIMULATED counts among 10,000 people:", counts.to_dict())
n_cough = counts["F∩C"] + counts["F̄∩C"]
print(f"P(C) ≈ {n_cough} / 10,000 = {n_cough / 10_000:.4f};  P(F | C) ≈ {counts['F∩C']} / {n_cough} = {counts['F∩C'] / n_cough:.4f}")

🔍 **What just happened?** In the **mosaic plot** each column's width is the probability of a flu status (0.5 each) and
each block's height is the conditional probability of cough within that column ($P(C \mid F) = 0.6$, $P(C \mid
\overline{F}) = 0.3$), so every block's **area** is a joint probability — width × height is exactly the law of total
probability. $P(C)$ is the total dark area, $0.30 + 0.15$. In 10,000 simulated people 4,586 cough, and 2,999 of them
(0.654) have the flu — close to 2/3; conditioning in data is just counting inside the subgroup.

### 📘 Lecture notes — conditional probability and the sum rule

**Conditional probability restricts the sample space.** For $(\Omega, \mathcal{A}, P)$ and $A, B \in \mathcal{A}$ with
$P(B) > 0$: $P(A \mid B) = P(A \cap B)/P(B)$. Using the axiom of total probability, $P(A \mid \Omega) = P(A \cap
\Omega)/P(\Omega) = P(A)/1$.

*Coins:* $A_{\text{allheads}} \subseteq A_{\text{firstheads}} \Rightarrow A_{\text{allheads}} \cap A_{\text{firstheads}}
= A_{\text{allheads}}$, so $P(A_{\text{allheads}} \mid A_{\text{firstheads}}) = \frac{1/4}{1/2} = \frac12$ and
$P(A_{\text{firstheads}} \mid A_{\text{allheads}}) = \frac{1/4}{1/4} = 1$.

*Flu/cough table:* $P(F \cap C) = 0.3$, $P(F \cap \overline{C}) = 0.2$, $P(\overline{F} \cap C) = 0.15$,
$P(\overline{F} \cap \overline{C}) = 0.35$ (sum 1). $P(C) = P(C \cap F) + P(C \cap \overline{F}) = 0.45$;
$P(F \mid C) = 0.3/0.45 = 2/3$ = (people with cough and flu)/(people with cough).

**Sum rule** (partition $A_1, \ldots, A_n$: pairwise disjoint, $\bigcup_i A_i = \Omega$): $P(B) = \sum_i P(B \cap A_i)$.
**Law of total probability:** $P(B) = \sum_i P(B \mid A_i)P(A_i)$.

> 🎓 **Exam tip.** In a table question, first write the margins (row and column totals) — almost every conditional
> probability is then "one cell divided by one margin". And check that the table sums to 1.

✏️ **Your turn.** (1) From the flu/cough table compute $P(C \mid F)$, $P(C \mid \overline{F})$ and $P(\overline{F} \mid
\overline{C})$. (2) Verify the law of total probability for $P(F)$ using the partition $\{C, \overline{C}\}$.
(3) Draw the mosaic plot the other way round (columns = cough status). Which picture makes $P(F \mid C)$ easiest to read?

---
## 3 · The product rule and the chain rule

> 📍 **Where we are.** The definition of conditional probability can be turned around to *build* joint probabilities
> step by step. This is how we compute the probability of a sequence of events — three pups, three bases, three
> generations.

**Product rule.** Multiplying the definition by $P(B)$ (if $P(B) > 0$):

$$P(A \cap B) = P(A \mid B)\,P(B) = P(B \mid A)\,P(A).$$

*In words:* the probability that both happen is the probability that one happens times the probability that the other
happens *given* the first; either order works (the second form needs $P(A) > 0$). It is only an algebraic rearrangement
of the definition, but it is used constantly.

**Chain rule.** Apply the product rule repeatedly to $n$ events $D_1, D_2, \ldots, D_n$ (notice: there is no condition
on these events — they need not be independent or special in any way). Peel off the last event, then the next-to-last,
and so on:

$$\begin{aligned}
P(D_1 \cap D_2 \cap \cdots \cap D_n)
&= P(D_n \mid D_1 \cap \cdots \cap D_{n-1})\,P(D_1 \cap \cdots \cap D_{n-1})\\
&= P(D_n \mid D_1 \cap \cdots \cap D_{n-1})\,P(D_{n-1} \mid D_1 \cap \cdots \cap D_{n-2})\,P(D_1 \cap \cdots \cap D_{n-2})\\
&= \cdots\\
&= P(D_n \mid D_1 \cap \cdots \cap D_{n-1}) \cdots P(D_3 \mid D_1 \cap D_2)\,P(D_2 \mid D_1)\,P(D_1)\\
&= P(D_1)\,P(D_2 \mid D_1)\,P(D_3 \mid D_1 \cap D_2) \cdots P(D_n \mid D_1 \cap \cdots \cap D_{n-1}).
\end{aligned}$$

*In words:* the probability of a whole sequence of events is the probability of the first, times the probability of
the second given the first, times the probability of the third given the first two, and so on. Each line applies the
product rule with $A = $ the last remaining event and $B = $ the intersection of all the earlier ones; the last line
just reverses the order of the factors.

**Example.** A litter of 10 mouse pups contains 4 that carry a knockout allele (we only learn which ones after
genotyping). We pick 3 pups at random, one after another, without replacement. Let $D_i$ = "the $i$-th pup picked is a
carrier". Then $P(D_1) = 4/10$; given $D_1$, 3 carriers remain among 9 pups, so $P(D_2 \mid D_1) = 3/9$; and
$P(D_3 \mid D_1 \cap D_2) = 2/8$. The chain rule gives $P(D_1 \cap D_2 \cap D_3) = \frac{4}{10} \cdot \frac39 \cdot
\frac28 = \frac{1}{30}$.

In [ ]:
# ── 3.1 Chain rule: three carriers in a row from a litter of 10 (exact, enumerated, SIMULATED) ──
carrier = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0])               # pups 0–3 carry the knockout allele
chain = Fraction(4, 10) * Fraction(3, 9) * Fraction(2, 8)
print("chain rule: P(D1 ∩ D2 ∩ D3) = 4/10 · 3/9 · 2/8 =", chain)

ordered_triples = list(itertools.permutations(range(10), 3))      # all equally likely ordered picks
favourable = sum(all(carrier[list(t)]) for t in ordered_triples)
print(f"enumeration: {favourable} of {len(ordered_triples)} ordered triples are all carriers →",
      Fraction(favourable, len(ordered_triples)))

rng = np.random.default_rng(SEED + 31)
picks = np.array([rng.permutation(10)[:3] for _ in range(60_000)])
print(f"SIMULATED (60,000 experiments): {carrier[picks].all(axis=1).mean():.4f}   exact {float(chain):.4f}")
print("if we picked WITH replacement (independent picks): (4/10)³ =", Fraction(4, 10) ** 3)

🔍 **What just happened?** Three routes, one answer: the chain rule gives $1/30 \approx 0.0333$; listing all
$10 \times 9 \times 8 = 720$ equally likely ordered picks finds $4 \times 3 \times 2 = 24$ favourable ones
($24/720 = 1/30$); and 60,000 simulated experiments give a frequency close to 0.0333. Each factor of the chain rule
is a *conditional* probability that reflects what earlier picks removed from the litter. Had we put each pup back
before the next pick, the picks would be independent and the answer would be $(4/10)^3 = 8/125 = 0.064$ — almost twice
as large.

The chain rule also gives the probability of a **DNA sequence**: $P(x_1 x_2 \ldots x_n) = P(x_1)\,P(x_2 \mid x_1)\,P(x_3
\mid x_1 x_2) \cdots$. A **Markov chain** ([Module 20 · ⛓️ Graphical models: Bayesian networks and Markov chains](20_bayesian_networks_markov_chains.ipynb)) adds one *assumption* — each base depends only on the previous one,
$P(x_t \mid x_1 \ldots x_{t-1}) = P(x_t \mid x_{t-1})$ — which makes the factors easy to estimate. The toolkit's
`BACKGROUND_TRANSITIONS` holds such probabilities for human DNA.

In [ ]:
# ── 3.2 The chain rule for a DNA string under a first-order Markov model ──
idx = {b: i for i, b in enumerate("ACGT")}
transitions = pd.DataFrame(BACKGROUND_TRANSITIONS, index=list("ACGT"), columns=list("ACGT"))
print("P(next base | current base) in background DNA (rows = current):")
print(transitions)


def sequence_probability(seq: str, P_next: np.ndarray, p_first: float = 0.25) -> float:
    """Chain rule with the Markov assumption: P(x1) · Π_t P(x_t | x_{t-1})."""
    prob = p_first
    for prev, nxt in zip(seq[:-1], seq[1:]):
        prob *= P_next[idx[prev], idx[nxt]]
    return prob


for s in ["ACGT", "AGCT", "CGCG", "GCGC"]:
    factors = " · ".join(f"{BACKGROUND_TRANSITIONS[idx[a], idx[b]]:.2f}" for a, b in zip(s[:-1], s[1:]))
    print(f"P({s}) = 0.25 · {factors} = {sequence_probability(s, BACKGROUND_TRANSITIONS):.6f}")

🔍 **What just happened?** Each row of the table is a conditional distribution $P(\text{next} \mid \text{current})$ and
sums to 1. The chain rule multiplies one starting probability (0.25) and one conditional probability per step. "CGCG"
(0.000400) is about 3 times less probable than "GCGC" (0.001250), and "ACGT" (0.000840) about 6 times less probable
than "AGCT" (0.005437), although each pair contains the same letters — because the factor $P(G \mid C) = 0.08$ is small: **C followed by G is rare in background human DNA**. Section 6 explains why and shows
that this makes neighbouring bases *dependent*.

### 📘 Lecture notes — the product rule and the chain rule

**Product rule** (an algebraic rearrangement of the definition; $P(B) > 0$): $P(A \cap B) = P(A \mid B)P(B)$, and
symmetrically $P(A \cap B) = P(B \mid A)P(A)$.

**Chain rule** for any events $D_1, \ldots, D_n$ (no conditions on them):

$$P(D_1 \cap \cdots \cap D_n) = P(D_1)\,P(D_2 \mid D_1)\,P(D_3 \mid D_1 \cap D_2) \cdots P(D_n \mid D_1 \cap \cdots \cap D_{n-1}).$$

*In words:* the first event, then each further event given all the earlier ones. Derivation: apply the product rule to
$D_n$ and $D_1 \cap \cdots \cap D_{n-1}$, then to $D_{n-1}$ and $D_1 \cap \cdots \cap D_{n-2}$, and so on until only
$P(D_1)$ is left.

*Worked example:* 3 carriers picked from 10 pups with 4 carriers: $\frac{4}{10}\cdot\frac39\cdot\frac28 = \frac1{30}$.

> 🎓 **Exam tip.** The chain rule holds in *any* order of the events and never needs independence. If an exam asks you to
> "simplify" it, the simplification always comes from an extra assumption (independence, a Markov property, a
> network) — say which one.

✏️ **Your turn.** (1) What is the probability that the 3 picked pups contain **no** carrier? Use the chain rule, then check
by enumeration. (2) Write the chain rule for $P(D_1 \cap D_2 \cap D_3)$ in the order $D_3, D_1, D_2$ and check that it gives
the same 1/30. (3) Compute $P(\texttt{ACGCGT})$ with `sequence_probability`.

---
## 4 · Independence

> 📍 **Where we are.** Sometimes learning $B$ tells us nothing about $A$: the second toss of a fair coin does not care
> about the first. This special — and enormously useful — situation is called independence.

**Definition.** Events $A$ and $B$ are **independent** if

$$P(A \cap B) = P(A) \cdot P(B).$$

*In words:* the probability that both happen is the product of their separate probabilities. We also know that, for
*any* pair of events, $P(A \cap B) = P(B \mid A)P(A) = P(A \mid B)P(B)$ (product rule). Hence, if $A$ and $B$ are
independent and $P(B) > 0$:

$$P(A \mid B) = \frac{P(A \cap B)}{P(B)} = \frac{P(A)\,P(B)}{P(B)} = P(A).$$

*In words:* learning that $B$ happened does not change the probability of $A$. Conversely, if $P(A \mid B) = P(A)$
then $P(A \cap B) = P(A \mid B)P(B) = P(A)P(B)$. So for $P(B) > 0$ the two statements are **equivalent**; the product
form is the official definition because it is symmetric and also works when $P(B) = 0$.

Examples: the two tosses of a fair coin ($P(HH) = \tfrac14 = \tfrac12 \cdot \tfrac12$); on a fair die,
$A = \{1, 2\}$ and $B = \{2, 4, 6\}$ ($P(A \cap B) = P(\{2\}) = \tfrac16 = \tfrac13 \cdot \tfrac12$); and **Mendel's
second law** (independent assortment): in the dihybrid cross of Module 16, $P(\text{yellow} \cap \text{round}) =
\tfrac{9}{16} = \tfrac34 \cdot \tfrac34$.

⚠️ **Common confusion: independent ≠ disjoint.** Disjoint events with positive probabilities are always *dependent*: if
$A \cap B = \emptyset$ then $P(A \cap B) = 0 \ne P(A)P(B)$. Knowing that $B$ happened tells you for certain that $A$ did
**not** — the opposite of "no information".

In [ ]:
# ── 4.1 Testing independence exactly ──
def independent(P, A: set, B: set) -> bool:
    """Events A and B are independent iff P(A ∩ B) = P(A) P(B)."""
    return P(A & B) == P(A) * P(B)


Omega_die = {1, 2, 3, 4, 5, 6}
P_die = lambda event: Fraction(len(set(event) & Omega_die), 6)
A_last = {w for w in Omega2 if w[1] == "H"}
cases = [("coins: first heads, last heads", P2, A_first, A_last),
         ("die: {1,2} and {2,4,6}", P_die, {1, 2}, {2, 4, 6}),
         ("die: even and ≥ 4", P_die, {2, 4, 6}, {4, 5, 6}),
         ("die: {1} and {2} (disjoint)", P_die, {1}, {2})]
for name, P, A, B in cases:
    print(f"{name:32s} P(A∩B) = {str(P(A & B)):4s}  P(A)P(B) = {str(P(A) * P(B)):5s}  independent: {independent(P, A, B)}"
          f"   P(A|B) = {conditional(P, A, B)} vs P(A) = {P(A)}")

# Mendel's second law: yellow and round in the dihybrid cross
gametes = ["".join(g) for g in itertools.product("Yy", "Rr")]
Omega16 = set(itertools.product(gametes, gametes))
P16 = lambda event: Fraction(len(set(event) & Omega16), 16)
yellow = {w for w in Omega16 if "Y" in w[0][0] + w[1][0]}
round_ = {w for w in Omega16 if "R" in w[0][1] + w[1][1]}
print(f"\nMendel: P(yellow ∩ round) = {P16(yellow & round_)} = P(yellow)·P(round) = {P16(yellow)}·{P16(round_)}:",
      independent(P16, yellow, round_))
print("flu table: P(F | C) = 2/3 but P(F) = 1/2 → flu and cough are dependent")

🔍 **What just happened?** The two coin tosses are independent ($\tfrac14 = \tfrac12 \cdot \tfrac12$), and so are
$\{1, 2\}$ and the even faces of a die: knowing the roll is even leaves $P(\{1, 2\}) = 1/3$ unchanged. "Even" and
"$\ge 4$" are dependent: $P(A \cap B) = 1/3 > 1/4$, and learning "$\ge 4$" raises the probability of "even" from $1/2$ to
$2/3$. The disjoint events $\{1\}$ and $\{2\}$ are dependent ($0 \ne 1/36$; given $\{2\}$, $\{1\}$ is impossible).
Mendel's yellow and round seeds are independent ($9/16 = 3/4 \cdot 3/4$) because the two genes sit on different
chromosomes and their gametes combine every allele of one gene with every allele of the other equally often. And flu
and cough are dependent: a cough raises $P(F)$ from 0.5 to 2/3.

> 🧬 **Biology check.** Mendel's second law fails for genes that are close together on the same chromosome: their alleles
> travel together unless a crossover separates them (Module 16, Section 1.3). Such **linked** genes are dependent —
> Section 6 measures that dependence.

✏️ **Your turn.** (1) Prove that if $A$ and $B$ are independent, so are $A$ and $\overline{B}$. (Hint: $P(A) = P(A \cap B)
+ P(A \cap \overline{B})$.) Check it on the die example in code. (2) Find another pair of independent events on the fair
die, neither of which is $\emptyset$ or $\Omega$.

---
## 5 · Conditional independence

> 📍 **Where we are.** Independence is about what we know *before* any information. Often two events are linked only
> through a third — a hidden cause — and become independent once that cause is known. This is **conditional
> independence**, the assumption behind naïve Bayes, Bayesian networks and hidden Markov models.

**Definition.** $A$ and $B$ are **conditionally independent given $C$** (with $P(C) > 0$) if

$$P(A \cap B \mid C) = P(A \mid C)\cdot P(B \mid C).$$

*In words:* inside the restricted world where $C$ happened, $A$ and $B$ are independent — the product rule of
independence holds with every probability conditioned on $C$.

### 5.1 A box with two coins

A box contains two coins: a **regular** coin ($P(H) = P(T) = \tfrac12$) and a **fake two-headed** coin ($P(H) = 1$). We
choose a coin at random and flip it twice. Define the events

* $A$ — the first toss is heads (so $\overline{A}$: tails);
* $B$ — the second toss is heads ($\overline{B}$: tails);
* $C$ — the regular coin was selected ($\overline{C}$: the fake, biased coin).

The conditional probabilities are read off the description:

| | given $C$ (regular) | given $\overline{C}$ (two-headed) |
|---|---|---|
| $P(A \mid \cdot)$ | 1/2 | 1 |
| $P(B \mid \cdot)$ | 1/2 | 1 |
| $P(A \cap B \mid \cdot)$ | 1/4 | 1 |

with $P(C) = P(\overline{C}) = 1/2$. Since $P(A \cap B \mid C) = \tfrac14 = P(A \mid C) \cdot P(B \mid C)$ and
$P(A \cap B \mid \overline{C}) = 1 = P(A \mid \overline{C}) \cdot P(B \mid \overline{C})$, **$A$ and $B$ are conditionally
independent given which coin was chosen**. But are they also independent without conditioning — does $P(A \cap B) =
P(A) \cdot P(B)$ hold? Use the sum rule with the partition $\{C, \overline{C}\}$ and then the product rule:

$$P(A) = P(A \cap C) + P(A \cap \overline{C}) = P(A \mid C)P(C) + P(A \mid \overline{C})P(\overline{C}) = \tfrac12 \cdot \tfrac12 + 1 \cdot \tfrac12 = \tfrac34,$$

*In words:* the first toss is heads either with the regular coin (probability 1/2 of being chosen, then 1/2 of heads)
or with the fake coin (1/2, then certainly heads). By the same logic $P(B) = 3/4$. For the intersection we use the same
trick, plus the conditional independence:

$$\begin{aligned}
P(A \cap B) &= P(A \cap B \mid C)P(C) + P(A \cap B \mid \overline{C})P(\overline{C})\\
&= P(A \mid C)P(B \mid C)P(C) + P(A \mid \overline{C})P(B \mid \overline{C})P(\overline{C})\\
&= \tfrac12 \cdot \tfrac12 \cdot \tfrac12 + 1 \cdot 1 \cdot \tfrac12 = \tfrac18 + \tfrac12 = \tfrac58 = \tfrac{10}{16}.
\end{aligned}$$

*In words:* both tosses are heads either with the regular coin ($\tfrac12 \times \tfrac14$) or with the fake coin
($\tfrac12 \times 1$). But $P(A) \cdot P(B) = \tfrac34 \cdot \tfrac34 = \tfrac{9}{16} \ne \tfrac{10}{16}$: **the coin
flips are not independent.**

In [ ]:
# ── 5.1 The two-coin box: exact enumeration of the probability space ──
P_heads = {"regular": Fraction(1, 2), "fake": Fraction(1)}
weights = {}                                                  # outcome (coin, toss1, toss2) → probability
for coin in P_heads:
    for t1, t2 in itertools.product("HT", repeat=2):
        p1 = P_heads[coin] if t1 == "H" else 1 - P_heads[coin]
        p2 = P_heads[coin] if t2 == "H" else 1 - P_heads[coin]
        weights[(coin, t1, t2)] = Fraction(1, 2) * p1 * p2     # chain rule: P(coin) P(t1 | coin) P(t2 | coin)
P_box = lambda event: sum((weights[w] for w in event), Fraction(0))
A = {w for w in weights if w[1] == "H"}; B = {w for w in weights if w[2] == "H"}
C = {w for w in weights if w[0] == "regular"}; notC = set(weights) - C
for name, cond in [("C (regular)", C), ("C̄ (fake)", notC)]:
    lhs, rhs = conditional(P_box, A & B, cond), conditional(P_box, A, cond) * conditional(P_box, B, cond)
    print(f"given {name:12s}: P(A∩B|·) = {lhs}   P(A|·)P(B|·) = {rhs}   conditionally independent: {lhs == rhs}")
print(f"\nP(A) = {P_box(A)}, P(B) = {P_box(B)}, P(A∩B) = {P_box(A & B)}, P(A)P(B) = {P_box(A) * P_box(B)}"
      f"  → independent: {independent(P_box, A, B)}")
print("P(B | A) =", conditional(P_box, B, A), "> P(B) =", P_box(B), "  and  P(C | A) =", conditional(P_box, C, A))

🔍 **What just happened?** We built the full probability space — 8 outcomes (coin, first toss, second toss) with
weights from the chain rule — and every number of the hand calculation came out: conditionally independent given
$C$ (1/4 = 1/4) and given $\overline{C}$ (1 = 1), yet $P(A \cap B) = 5/8 \ne 9/16 = P(A)P(B)$. The last line shows *why*:
a head on the first toss is evidence about the **coin**. It lowers the probability that we hold the regular coin from
1/2 to $P(C \mid A) = 1/3$, and therefore raises the probability of a second head from $P(B) = 3/4$ to
$P(B \mid A) = 5/6$. The tosses are linked through the hidden common cause. A simulation shows the same in data.

In [ ]:
# ── 5.2 The two-coin box, SIMULATED 200,000 times ──
rng = np.random.default_rng(SEED + 52)
n = 200_000
regular = rng.random(n) < 0.5                                     # C: the regular coin was drawn
p_h = np.where(regular, 0.5, 1.0)
first, second = rng.random(n) < p_h, rng.random(n) < p_h          # A, B: independent GIVEN the coin
print(f"P(A) ≈ {first.mean():.4f}  P(B) ≈ {second.mean():.4f}  P(A∩B) ≈ {(first & second).mean():.4f}"
      f"  P(A)P(B) ≈ {first.mean() * second.mean():.4f}   (exact 0.625 vs 0.5625)")
for name, mask in [("regular coin", regular), ("fake coin", ~regular)]:
    a, b = first[mask], second[mask]
    print(f"within {name:12s}: P(A∩B|·) ≈ {(a & b).mean():.4f}   P(A|·)P(B|·) ≈ {a.mean() * b.mean():.4f}")

🔍 **What just happened?** In 200,000 simulated draws, $P(A \cap B) \approx 0.625$ while $P(A)P(B) \approx 0.5625$ —
dependent; within each kind of coin the joint frequency matches the product of the separate frequencies (≈ 0.25 vs
≈ 0.25 for the regular coin; exactly 1 vs 1 for the fake one). This is the pattern of a **hidden common cause**. In
biology: two measurements of the same patient (two biomarkers, two replicate assays) are correlated because they share
the patient's disease state, yet they may be independent *given* that state.

### 5.2 A second example: a probability table for $A$, $B$, $C$

Suppose three events have the following joint probabilities (each row splits $C$ or $\overline{C}$ into the four
combinations of $A$ and $B$):

| | $A \cap B$ | $A \cap \overline{B}$ | $\overline{A} \cap B$ | $\overline{A} \cap \overline{B}$ | row total |
|---|---|---|---|---|---|
| $C$ | 0.1 | 0.2 | 0.2 | 0.1 | 0.6 |
| $\overline{C}$ | 0.1 | 0.1 | 0.1 | 0.1 | 0.4 |
| column total | 0.2 | 0.3 | 0.3 | 0.2 | 1.0 |

Given $\overline{C}$: $P(A \cap B \mid \overline{C}) = \tfrac{0.1}{0.4} = 0.25$, $P(A \mid \overline{C}) =
\tfrac{0.1 + 0.1}{0.4} = 0.5$, $P(B \mid \overline{C}) = \tfrac{0.1 + 0.1}{0.4} = 0.5$, and $0.25 = 0.5 \times 0.5$ —
so $A$ and $B$ are conditionally independent **given $\overline{C}$**. Unconditionally: $P(A) = 0.2 + 0.3 = 0.5$,
$P(B) = 0.2 + 0.3 = 0.5$, but $P(A \cap B) = 0.2 \ne 0.25 = P(A)P(B)$ — **not independent**. What about the row $C$?

In [ ]:
# ── 5.3 The A–B–C table: check both conditional independences and independence (exact fractions) ──
tenth = Fraction(1, 10)
abc = pd.DataFrame([[1 * tenth, 2 * tenth, 2 * tenth, 1 * tenth], [1 * tenth, 1 * tenth, 1 * tenth, 1 * tenth]],
                   index=["C", "C̄"], columns=["A∩B", "A∩B̄", "Ā∩B", "Ā∩B̄"])
print("table sums to", abc.to_numpy().sum())
for row in ["C", "C̄"]:
    r = abc.loc[row]; p_row = r.sum()
    p_ab = r["A∩B"] / p_row
    p_a = (r["A∩B"] + r["A∩B̄"]) / p_row
    p_b = (r["A∩B"] + r["Ā∩B"]) / p_row
    print(f"given {row}: P(A∩B|{row}) = {p_ab}   P(A|{row})·P(B|{row}) = {p_a}·{p_b} = {p_a * p_b}"
          f"   → conditionally independent: {p_ab == p_a * p_b}")
col = abc.sum(axis=0)
pA, pB, pAB = col["A∩B"] + col["A∩B̄"], col["A∩B"] + col["Ā∩B"], col["A∩B"]
print(f"unconditional: P(A∩B) = {pAB}   P(A)·P(B) = {pA}·{pB} = {pA * pB}   → independent: {pAB == pA * pB}")

🔍 **What just happened?** Given $\overline{C}$ the factorisation holds ($1/4 = 1/2 \cdot 1/2$), but given $C$ it fails:
$P(A \cap B \mid C) = 0.1/0.6 = 1/6$, while $P(A \mid C)P(B \mid C) = 0.5 \times 0.5 = 1/4$. And $A$, $B$ are not
independent overall ($1/5 \ne 1/4$). ⚠️ **Common confusion.** "Conditionally independent given $C$" in the strict sense
of the definition refers to one event $C$; when $C$ stands for a *variable* (which coin, which disease state), we
usually mean independence **given every value of it** — here given $C$ *and* given $\overline{C}$. This table satisfies
only the second, so the careful statement is: *$A$ and $B$ are conditionally independent given $\overline{C}$, but not
given $C$, and not independent.* It still makes the main point: conditional independence (given $\overline{C}$) does not
imply independence.

### 5.3 The converse fails too: independent but not conditionally independent

Consider again rolling a fair die and let $A = \{1, 2\}$, $B = \{2, 4, 6\}$, $C = \{1, 4\}$. Section 4 showed that $A$
and $B$ are independent. But given $C$: $P(A \mid C) = \tfrac12$ (only face 1 of $\{1, 4\}$), $P(B \mid C) = \tfrac12$
(only face 4), and $P(A \cap B \mid C) = P(\{2\} \mid C) = 0$ because $2 \notin C$. Thus $P(A \cap B \mid C) = 0 \ne
\tfrac14 = P(A \mid C) \cdot P(B \mid C)$.

In [ ]:
# ── 5.4 Independent, but not conditionally independent: the die ──
A_d, B_d, C_d = {1, 2}, {2, 4, 6}, {1, 4}
print("independent:", independent(P_die, A_d, B_d), f"(P(A∩B) = {P_die(A_d & B_d)} = {P_die(A_d)}·{P_die(B_d)})")
print(f"given C: P(A∩B|C) = {conditional(P_die, A_d & B_d, C_d)}   P(A|C)·P(B|C) = "
      f"{conditional(P_die, A_d, C_d)}·{conditional(P_die, B_d, C_d)} = {conditional(P_die, A_d, C_d) * conditional(P_die, B_d, C_d)}")
summary = pd.DataFrame({"independent?": ["no", "no", "yes"],
                        "conditionally independent?": ["yes (given C and given C̄)", "given C̄ only", "no (given C)"]},
                       index=["two-coin box", "A–B–C table", "die {1,2}, {2,4,6}, {1,4}"])
print("\n", summary.to_string())

🔍 **What just happened?** The die events are independent ($1/6 = 1/3 \cdot 1/2$), but inside $C = \{1, 4\}$ they are
mutually exclusive: $A \cap B \cap C = \emptyset$, so $P(A \cap B \mid C) = 0$ instead of $1/4$. The summary table
collects the three examples. **Independence does not imply conditional independence, and conditional independence does
not imply independence.** Each must be checked separately — a fact that will matter in every graphical model you meet.

### 📘 Lecture notes — independence and conditional independence

**Independence:** $P(A \cap B) = P(A)\cdot P(B)$. Since $P(A \cap B) = P(B \mid A)P(A) = P(A \mid B)P(B)$ always, for
independent $A, B$ (with $P(B) > 0$) $P(A \mid B) = P(A)$. ⚠️ Disjoint events with positive probability are dependent.

**Conditional independence:** $P(A \cap B \mid C) = P(A \mid C)\cdot P(B \mid C)$.

**Example 1 — the coin box** (regular coin and two-headed coin; pick one, flip twice; $A$ = first heads, $B$ = second
heads, $C$ = regular coin): $P(A \mid C) = P(B \mid C) = \tfrac12$, $P(A \cap B \mid C) = \tfrac14$;
$P(A \mid \overline{C}) = P(B \mid \overline{C}) = P(A \cap B \mid \overline{C}) = 1$; $P(C) = P(\overline{C}) = \tfrac12$.
Conditionally independent given $C$ and given $\overline{C}$. Sum + product rules: $P(A) = \tfrac12\cdot\tfrac12 +
1\cdot\tfrac12 = \tfrac34 = P(B)$; $P(A \cap B) = \tfrac12\cdot\tfrac12\cdot\tfrac12 + 1\cdot1\cdot\tfrac12 = \tfrac58 =
\tfrac{10}{16} \ne \tfrac{9}{16}$ → **not independent**.

**Example 2 — the A–B–C table** (rows $C$: 0.1, 0.2, 0.2, 0.1; $\overline{C}$: 0.1, 0.1, 0.1, 0.1): given
$\overline{C}$, $P(A \cap B \mid \overline{C}) = 0.25 = 0.5 \times 0.5$; given $C$, $1/6 \ne 1/4$; $P(A \cap B) = 0.2
\ne 0.25 = P(A)P(B)$.

**Example 3 — the die** ($A = \{1,2\}$, $B = \{2,4,6\}$, $C = \{1,4\}$): $P(A \cap B) = \tfrac16 = P(A)P(B)$
(independent), but $P(A \cap B \mid C) = 0 \ne \tfrac14 = P(A \mid C)P(B \mid C)$.

**Conclusion:** conditional independence does NOT imply independence, and independence does NOT imply conditional
independence.

> 🎓 **Exam tip.** To *disprove* (conditional) independence, one counterexample cell is enough; to *prove* conditional
> independence given a variable, check the factorisation for every value of the conditioning variable.

### 🌉 Bridge — conditional independence is how probabilistic models stay small

A full joint table of $n$ yes/no variables has $2^n - 1$ free numbers — over a million for 20 genes. Every practical
probabilistic model in biology tames this with **conditional-independence assumptions** of exactly the kind in the
coin box:

* **Naïve Bayes** (Section 10, [BiologyToML 08 · k-nearest neighbours and naïve Bayes](../../../../ml/tutorial/BiologyToML/notebooks/08_knn_naive_bayes.ipynb)): features (genes, words) are independent *given the class* (cell type, spam).
* **Bayesian networks** ([Module 20 · ⛓️ Graphical models: Bayesian networks and Markov chains](20_bayesian_networks_markov_chains.ipynb), [BiologyToML 15 · Bayesian networks: representation](../../../../ml/tutorial/BiologyToML/notebooks/15_bayesian_networks_representation.ipynb)): each variable is independent of its non-descendants given its parents — the
  coin box is the network "coin → toss 1, coin → toss 2".
* **Hidden Markov models** ([Module 20 · ⛓️ Graphical models: Bayesian networks and Markov chains](20_bayesian_networks_markov_chains.ipynb), [BiologyToML 19 · Hidden Markov models: inference](../../../../ml/tutorial/BiologyToML/notebooks/19_hmm_inference.ipynb)): the observed bases are independent given the hidden gene structure.

The coin box is also the simplest example of why such models *must* include hidden variables: two tosses that look
dependent become independent once the hidden coin is in the model.

✏️ **Your turn.** (1) In the coin box, compute $P(C \mid A \cap B)$ — how sure are you that you hold the regular coin
after two heads? Then simulate it. (2) Put **three** coins in the box (regular, two-headed, two-tailed). Are the two
tosses still conditionally independent given the coin? Are they independent?

---
## 6 · Are neighbouring nucleotides independent?

> 📍 **Where we are.** Independence is a modelling assumption, not a law of nature. DNA gives a clean test case:
> consider the nucleotides A/C/G/T at two adjacent positions in a genome. Are they independent in background
> sequence? In a stop codon?

**Background model.** In the simplest model of "background" DNA every base is equally likely at each locus and loci are
independent: $P(A) = P(C) = P(G) = P(T) = \tfrac14$ at locus 1 and at locus 2, so
$P(\text{locus 1} = A \cap \text{locus 2} = A) = \tfrac14 \cdot \tfrac14 = \tfrac1{16}$. We can check whether a sequence
looks like this by comparing the frequency of each **dinucleotide** with the product of the single-base frequencies.

In [ ]:
# ── 6.1 Dinucleotide frequencies vs products of base frequencies (SIMULATED sequences) ──
def dinucleotide_check(seq: str) -> tuple:
    """Joint table P(x_t, x_{t+1}) of adjacent bases and the table of products P(x_t) P(x_{t+1})."""
    pairs = pd.crosstab(pd.Series(list(seq[:-1]), name="base t"), pd.Series(list(seq[1:]), name="base t+1"),
                        normalize=True)
    product = np.outer(pairs.sum(axis=1), pairs.sum(axis=0))
    return pairs, pd.DataFrame(product, index=pairs.index, columns=pairs.columns)


iid_dna = simulate_dna(100_000, seed=61)                               # independent, uniform bases
markov_dna = simulate_dna(50_000, transitions=BACKGROUND_TRANSITIONS, seed=62)   # human-like background
for name, seq in [("independent uniform model", iid_dna), ("human-like background (Markov)", markov_dna)]:
    joint_tab, product_tab = dinucleotide_check(seq)
    ratio = joint_tab / product_tab
    print(f"{name}: P(AA) = {joint_tab.loc['A', 'A']:.4f} (1/16 = 0.0625);  largest |joint − product| = "
          f"{(joint_tab - product_tab).abs().to_numpy().max():.4f}")
    print(f"   observed/expected for CG = {ratio.loc['C', 'G']:.2f}, for GC = {ratio.loc['G', 'C']:.2f}")

🔍 **What just happened?** In the independent model the dinucleotide $AA$ has frequency 0.0614 ≈ 1/16 and every
joint frequency equals the product of the base frequencies up to sampling noise (largest difference 0.0015); the
observed/expected ratio of $CG$ is 0.99. In the human-like background model, $CG$ appears at only about a third of the
frequency that independence predicts (observed/expected 0.34), while $GC$ is at 1.02, and the largest gap between a
joint frequency and the product of marginals is 0.039:
**neighbouring bases are dependent**. This "CpG depletion" is real: a methylated C in a CpG often mutates to T, so over
evolutionary time CpGs disappear from the genome except in **CpG islands** around gene promoters ([Module 20 · ⛓️ Graphical models: Bayesian networks and Markov chains](20_bayesian_networks_markov_chains.ipynb) detects them
with Markov chains).

### 6.1 Stop codons

Now look at the three **stop codons** TAG, TAA and TGA (DNA alphabet), treated as equally likely. Are positions 1 and 2
independent? Positions 2 and 3? The combinations of positions 1–2 are TA (from TAG, TAA) and TG (from TGA), so
$P(\text{1st} = T) = 1$, $P(\text{2nd} = A) = 2/3$, $P(\text{2nd} = G) = 1/3$, and the joint $P(\text{1st} = T \cap
\text{2nd} = A) = 2/3 = 1 \times 2/3$.

In [ ]:
# ── 6.2 Joint and marginal base frequencies in the three stop codons (typed in) ──
stops = pd.DataFrame([list(c) for c in ["TAG", "TAA", "TGA"]], columns=["pos1", "pos2", "pos3"])
print(stops.T.to_string(header=["TAG", "TAA", "TGA"]))
for a, b in [("pos1", "pos2"), ("pos2", "pos3")]:
    joint_ab = pd.crosstab(stops[a], stops[b], normalize=True)
    prod_ab = pd.DataFrame(np.outer(joint_ab.sum(axis=1), joint_ab.sum(axis=0)), index=joint_ab.index, columns=joint_ab.columns)
    print(f"\n{a} × {b}: joint probabilities\n{joint_ab.round(3)}\nproducts of the marginals\n{prod_ab.round(3)}")
    print("independent:", np.allclose(joint_ab, prod_ab))
p3G = (stops.pos3 == "G").mean()
p3G_given_2A = (stops.pos3[stops.pos2 == "A"] == "G").mean()
p3A_given_2G = (stops.pos3[stops.pos2 == "G"] == "A").mean()
print(f"\nP(3rd = G | 2nd = A) = {p3G_given_2A:.3f}  vs  P(3rd = G) = {p3G:.3f}")
print(f"P(3rd = A | 2nd = G) = {p3A_given_2G:.3f}  vs  P(3rd = A) = {(stops.pos3 == 'A').mean():.3f}")

🔍 **What just happened?** For positions 1 and 2 the joint table equals the product of the marginals
($P(T \cap A) = 2/3 = 1 \cdot 2/3$ and $P(T \cap G) = 1/3 = 1 \cdot 1/3$): independent. ⚠️ But this independence is
*trivial*: position 1 is always T, and an event of probability 1 is independent of every event — it carries no
information. For positions 2 and 3 the tables differ (e.g. $P(\text{2nd} = A \cap \text{3rd} = A) = 1/3$ vs product
$2/3 \cdot 2/3 = 4/9$), and the conditional probabilities make it vivid: $P(\text{3rd} = G \mid \text{2nd} = A) = 1/2$
(TAG, TAA) but $P(\text{3rd} = G) = 1/3$; $P(\text{3rd} = A \mid \text{2nd} = G) = 1$ (TGA) but $P(\text{3rd} = A) =
2/3$. Neither marginal equals the conditional, so **positions 2 and 3 of a stop codon are not independent** — TGG
codes for tryptophan and TGT, TGC for cysteine, so a stop codon with G in position 2 *must* end in A.

### 6.2 Factors that break the independence of nucleotides

* **Coding regions** — start and stop codons, codon usage, splice sites (introns almost always begin with GT and end
  with AG), anything related to gene structure.
* **Regulatory regions** — transcription-factor **binding sites** are short sequence motifs in which some positions are
  nearly fixed and others vary; a sequence logo shows them as stacks of letters.
* **Linkage disequilibrium (LD)** — the non-random association of alleles at different genetic loci within a
  population. Alleles at nearby SNPs travel together on haplotypes until recombination separates them (Module 16,
  Section 1.3), so their joint frequency differs from the product of their allele frequencies.

In [ ]:
# ── 6.3 Linkage disequilibrium: dependence between alleles at two SNPs (ILLUSTRATIVE haplotype frequencies) ──
haplotypes = pd.DataFrame([[0.45, 0.05], [0.15, 0.35]], index=["A", "a"], columns=["B", "b"])   # P(allele at SNP1 ∩ SNP2)
pA, pB = haplotypes.loc["A"].sum(), haplotypes["B"].sum()
D = haplotypes.loc["A", "B"] - pA * pB                              # the LD coefficient
print(f"P(A) = {pA:.2f}, P(B) = {pB:.2f}, P(A ∩ B) = {haplotypes.loc['A', 'B']:.2f} vs P(A)P(B) = {pA * pB:.2f}  →  D = {D:.2f}")
print(f"P(B | A) = {haplotypes.loc['A', 'B'] / pA:.2f}  vs  P(B) = {pB:.2f}")

generations = np.arange(0, 201)
fig, ax = plt.subplots(figsize=(6.4, 3.3))
for r in [0.001, 0.01, 0.1, 0.5]:                                  # recombination fraction between the SNPs
    ax.plot(generations, D * (1 - r) ** generations, label=f"r = {r}")
ax.set_xlabel("generations of random mating"); ax.set_ylabel("LD coefficient D")
ax.set_title("recombination erodes LD: D_t = (1 − r)^t · D_0"); ax.legend()
plt.tight_layout(); plt.show()

🔍 **What just happened?** With these illustrative haplotype frequencies, allele $A$ (frequency 0.5) and allele $B$
(0.6) occur together on 45 % of chromosomes instead of the 30 % that independence predicts; the **LD coefficient**
$D = P(A \cap B) - P(A)P(B) = 0.15$ measures the departure, and knowing a chromosome carries $A$ raises the probability
of $B$ from 0.6 to 0.9. Each generation, recombination between the two loci (probability $r$) reshuffles alleles and
multiplies $D$ by $1 - r$: unlinked loci ($r = 0.5$) become independent within a few generations, while tightly linked
loci ($r = 0.001$) keep most of their LD for hundreds of generations.

### 🌉 Bridge — linkage disequilibrium in genome-wide association studies

LD is why **genome-wide association studies (GWAS)** work and why they are hard to interpret. A study genotypes a few
hundred thousand SNPs, not the causal variant itself; but because the causal variant is in LD with nearby genotyped
SNPs — $P(\text{SNP allele} \mid \text{causal allele}) \ne P(\text{SNP allele})$ — those SNPs show the association
too. The flip side: a significant SNP points to a whole LD block, and fine-mapping must decide which variant in the
block is causal. LD also breaks the "independent SNPs" assumption of naïve models ([BiologyToML 35 · Statistical genetics](../../../../ml/tutorial/BiologyToML/notebooks/35_statistical_genetics.ipynb)).

✏️ **Your turn.** (1) Using `dinucleotide_check`, find the dinucleotide with the *largest* observed/expected ratio in
`markov_dna`. (2) Copy `BACKGROUND_TRANSITIONS`, replace its C row by the CpG-island row $(0.17, 0.37, 0.27, 0.19)$ (inside CpG
islands $P(G \mid C) = 0.27$; [Module 20 · ⛓️ Graphical models: Bayesian networks and Markov chains](20_bayesian_networks_markov_chains.ipynb) uses the full island model), simulate with `simulate_dna(…, transitions=…)` and
compare the CG ratio.
(3) In the stop-codon table, compute $P(\text{2nd} = A \mid \text{3rd} = G)$.

---
## 7 · Conditional probability is itself a probability measure

> 📍 **Where we are.** We have used $P(\cdot \mid B)$ as if it were an ordinary probability. It is: for a fixed $B$ it
> satisfies all three axioms, so every theorem of Module 16 holds "given $B$". But it is a function of its *first*
> argument only — the conditioning event does not add up.

Fix an event $B$ with $P(B) > 0$ and define $Q(A) = P(A \mid B)$ for every event $A$. Then:

* **Non-negativity:** $Q(A) = P(A \cap B)/P(B) \ge 0$, a non-negative number divided by a positive one.
* **Normalisation:** $Q(\Omega) = P(\Omega \cap B)/P(B) = P(B)/P(B) = 1$.
* **Countable additivity:** if $A_1, A_2, \ldots$ are pairwise disjoint, so are $A_1 \cap B, A_2 \cap B, \ldots$, and
  $\left(\bigcup_i A_i\right) \cap B = \bigcup_i (A_i \cap B)$ (distributivity, Module 03). Hence

$$P\Big(\bigcup_{i=1}^{\infty} A_i \,\Big|\, B\Big) = \frac{P\big(\bigcup_i (A_i \cap B)\big)}{P(B)} = \frac{\sum_i P(A_i \cap B)}{P(B)} = \sum_{i=1}^{\infty} P(A_i \mid B).$$

*In words:* given $B$, probabilities of disjoint events still add, because intersecting with $B$ keeps them disjoint.
Consequently all the consequences of Module 16 hold given $B$: $P(\overline{A} \mid B) = 1 - P(A \mid B)$,
$P(A \cup A' \mid B) = P(A \mid B) + P(A' \mid B) - P(A \cap A' \mid B)$, and so on.

⚠️ **But not in the conditioning argument.** In general

$$P(A \mid B \cup C) \ne P(A \mid B) + P(A \mid C),$$

*In words:* conditioning on "$B$ or $C$" is not the sum of conditioning on each — even when $B$ and $C$ are disjoint.
Example: a deck of 52 cards; $A$ = the card is a heart (13 cards), $B$ = a heart from ace ("1") to 7 (7 cards),
$C$ = a heart from 8 to king (6 cards). Then $B \cup C = A$, so $P(A \mid B \cup C) = P(A \mid A) = 1$, but
$P(A \mid B) = P(A \mid C) = 1$ and $P(A \mid B) + P(A \mid C) = 2 \ne 1$.

In [ ]:
# ── 7.1 P(· | B) is a probability measure — and the card example (52 cards, typed in) ──
ranks = ["A", "2", "3", "4", "5", "6", "7", "8", "9", "10", "J", "Q", "K"]
deck = {(r, s) for r in ranks for s in "♥♦♣♠"}
P_card = lambda event: Fraction(len(set(event) & deck), 52)
hearts = {c for c in deck if c[1] == "♥"}
low_hearts = {c for c in hearts if ranks.index(c[0]) < 7}            # ace (1) … 7
high_hearts = hearts - low_hearts                                    # 8 … king
print(f"|A| = {len(hearts)}, |B| = {len(low_hearts)}, |C| = {len(high_hearts)};  B ∪ C = A: {low_hearts | high_hearts == hearts}")
print("P(A | B ∪ C) =", conditional(P_card, hearts, low_hearts | high_hearts),
      "  but P(A | B) + P(A | C) =", conditional(P_card, hearts, low_hearts) + conditional(P_card, hearts, high_hearts))

# the axioms for Q(·) = P(· | red) on 2,000 random pairs of disjoint events
red = {c for c in deck if c[1] in "♥♦"}
rng = np.random.default_rng(SEED + 71)
cards = sorted(deck)
ok_add, ok_compl = True, True
for _ in range(2000):
    labels = rng.integers(0, 3, 52)                                   # 0 → E1, 1 → E2, 2 → neither: E1, E2 disjoint
    E1 = {c for c, l in zip(cards, labels) if l == 0}; E2 = {c for c, l in zip(cards, labels) if l == 1}
    ok_add &= conditional(P_card, E1 | E2, red) == conditional(P_card, E1, red) + conditional(P_card, E2, red)
    ok_compl &= conditional(P_card, deck - E1, red) == 1 - conditional(P_card, E1, red)
print(f"Q(Ω) = P(Ω | red) = {conditional(P_card, deck, red)};  additivity on 2,000 disjoint pairs: {ok_add};  "
      f"complement rule: {ok_compl}")

🔍 **What just happened?** The deck has 13 hearts split into 7 low and 6 high ones. Conditioning on $B \cup C$ (= all
hearts) gives probability 1 of a heart, yet each of $P(A \mid B)$ and $P(A \mid C)$ is also 1, so their sum is 2 — a
"probability" larger than 1 would result if we added in the conditioning argument. In the first argument, by contrast,
everything works: $Q(\cdot) = P(\cdot \mid \text{red})$ gives the whole deck probability 1, adds exactly over 2,000
random pairs of disjoint events, and obeys the complement rule.

### 7.1 In general, $P(A \mid B) \ne P(B \mid A)$

The two conditional probabilities answer different questions and are usually different numbers:

| | $P(A \mid B)$ | $P(B \mid A)$ |
|---|---|---|
| coins: $A$ = all heads, $B$ = first heads | 1/2 | 1 |
| flu/cough: $A$ = flu, $B$ = cough | 0.30 / 0.45 = 0.667 | 0.30 / 0.50 = 0.6 |
| cards: $A$ = heart, $B$ = ace of hearts | 1 | 1/13 |

They are linked by $P(A \mid B)P(B) = P(A \cap B) = P(B \mid A)P(A)$ — which is Bayes' rule, the topic of the next
section. Confusing the two directions (the "prosecutor's fallacy") is among the commonest errors in medicine, genetics
and law.

### 📘 Lecture notes — properties of conditional probability

For fixed $B$ with $P(B) > 0$, $P(\cdot \mid B)$ satisfies the three axioms:

* non-negativity $P(A \mid B) \ge 0$;
* normalisation $P(\Omega \mid B) = 1$;
* countable additivity for mutually disjoint events $P(\bigcup_{i=1}^\infty A_i \mid B) = \sum_{i=1}^\infty P(A_i \mid B)$.

Please note that, in general, $P(A \mid B \cup C) \ne P(A \mid B) + P(A \mid C)$. *Card example:* 52 cards; $A$ = heart
(13 cards), $B$ = heart 1–7 (7 cards), $C$ = heart 8–K (6 cards): $B \cup C = A$, $P(A \mid B \cup C) = P(A \mid A) =
1$, $P(A \mid B) = P(A \mid C) = 1$, so $P(A \mid B) + P(A \mid C) = 2 \ne 1$.

In general $P(A \mid B) \ne P(B \mid A)$.

> 🎓 **Exam tip.** "Is $P(\cdot \mid B)$ a probability?" — yes, prove the three axioms. "Is $P(A \mid \cdot)$ a
> probability?" — no, give the card counterexample.

✏️ **Your turn.** (1) Prove $P(\overline{A} \mid B) = 1 - P(A \mid B)$ directly from the definition. (2) For the deck,
compute $P(\text{king} \mid \text{face card})$ and $P(\text{face card} \mid \text{king})$ (face cards: J, Q, K).

---
## 8 · Bayes' rule

> 📍 **Where we are.** Section 7 ended with $P(A \mid B)P(B) = P(B \mid A)P(A)$. Solving for $P(A \mid B)$ gives the
> most famous formula in probability, published posthumously for Thomas Bayes in 1763.

For a probability space $(\Omega, \mathcal{A}, P)$, suppose some event $B \in \mathcal{A}$ has already happened. The
probability that some event $A \in \mathcal{A}$ also happens is $P(A \mid B) = P(A \cap B)/P(B)$ (with $P(B) > 0$), and
for events $A$ and $B$ we have $P(A \cap B) = P(A \mid B)P(B) = P(B \mid A)P(A)$. Dividing the last equation by $P(B)$:

$$P(A \mid B) = \frac{P(B \mid A)\,P(A)}{P(B)} \qquad \textbf{(Bayes' rule)}.$$

*In words:* to turn "$B$ given $A$" into "$A$ given $B$", multiply by the probability of $A$ and divide by the
probability of $B$. The pieces have names:

* $P(A)$ — the **prior**: the initial belief in $A$ before seeing the evidence;
* $P(B \mid A)$ — the **likelihood**: how likely the evidence $B$ is if the hypothesis $A$ is true;
* $P(A \mid B)$ — the **posterior**: the updated belief after incorporating the evidence;
* $P(B)$ — the **evidence** (or marginal likelihood): the overall probability of what we observed.

The evidence is usually not given directly; the law of total probability with the partition $\{A, \overline{A}\}$
supplies it, giving **another form** of Bayes' rule:

$$P(A \mid B) = \frac{P(B \mid A)\,P(A)}{P(B)} = \frac{P(B \mid A)\,P(A)}{P(B \mid A)\,P(A) + P(B \mid \overline{A})\,P(\overline{A})}.$$

*In words:* the posterior is the share of "$B$ and $A$" among all the ways $B$ can happen — with $A$ or without it.

**Inverse probability reasoning.** Bayes' rule lets us go *backward* from an effect to its possible causes using known
likelihoods. Medicine and biology naturally provide **forward** probabilities $P(\text{effect} \mid \text{cause})$, e.g.
$P(\text{cough} \mid \text{flu})$ — how often flu patients cough, measured in flu patients. The doctor needs the
**backward** probability $P(\text{cause} \mid \text{effect})$, e.g. $P(\text{flu} \mid \text{cough})$.

**Example.** Let $A$ be the event that Bob has the flu and $B$ the event that Bob has a cough (illustrative numbers):

* prior $P(A) = 0.05$ — the background probability that Bob has the flu, regardless of symptoms;
* likelihood $P(B \mid A) = 0.8$ — the probability Bob coughs given he has the flu;
* $P(B \mid \overline{A}) = 0.2$ — the probability Bob coughs if he doesn't have the flu (colds, allergies …).

What is the probability Bob has the flu if he has a cough?

$$P(A \mid B) = \frac{(0.8)(0.05)}{(0.8)(0.05) + (0.2)(0.95)} = \frac{0.04}{0.04 + 0.19} \approx 0.174.$$

*In words:* only about 1 coughing person in 6 has the flu, because flu is rare (5 %) and many people cough for other
reasons. If instead flu is very common in Bob's situation — say half of his household is down with it, so the prior is
$P(A) = 0.5$ — then $P(A \mid B) = \frac{0.4}{0.4 + 0.1} = 0.8$.

In [ ]:
# ── 8.1 Bayes' rule for Bob's flu, with every intermediate number ──
def posterior(prior: float, likelihood: float, likelihood_if_not: float) -> float:
    """P(A | B) = P(B|A)P(A) / [P(B|A)P(A) + P(B|Ā)P(Ā)] — Bayes' rule with the law of total probability."""
    evidence = likelihood * prior + likelihood_if_not * (1 - prior)
    return likelihood * prior / evidence


for prior_flu in [0.05, 0.5]:
    num = 0.8 * prior_flu
    alt = 0.2 * (1 - prior_flu)
    print(f"prior {prior_flu}: P(B) = {num:.2f} + {alt:.2f} = {num + alt:.2f};  "
          f"P(flu | cough) = {num:.2f} / {num + alt:.2f} = {posterior(prior_flu, 0.8, 0.2):.4f}")

rng = np.random.default_rng(SEED + 81)
n = 100_000
flu = rng.random(n) < 0.05
cough = rng.random(n) < np.where(flu, 0.8, 0.2)
print(f"\nSIMULATED 100,000 people (prior 0.05): {cough.sum():,} cough, of whom {flu[cough].sum():,} have flu "
      f"→ {flu[cough].mean():.4f}")

🔍 **What just happened?** With prior 0.05 the evidence is $P(B) = 0.04 + 0.19 = 0.23$ and the posterior
$0.04/0.23 = 0.1739$; with prior 0.5 the evidence is $0.4 + 0.1 = 0.5$ and the posterior $0.8$. Same symptom, same
likelihoods, very different conclusions — **the prior matters**. The simulation of 100,000 people agrees: about 23,000
cough, and about 17 % of them have the flu. (⚠️ The prior is the probability of the *cause* — flu — before the symptom
is seen; it is not the probability of coughing.)

How does the posterior depend on the prior? A clean way to see it is the **odds form** of Bayes' rule. Writing
$\text{odds}(A) = P(A)/P(\overline{A})$ and dividing Bayes' rule for $A$ by Bayes' rule for $\overline{A}$ (the evidence
cancels):

$$\frac{P(A \mid B)}{P(\overline{A} \mid B)} = \frac{P(B \mid A)}{P(B \mid \overline{A})} \times \frac{P(A)}{P(\overline{A})}.$$

*In words:* posterior odds = likelihood ratio × prior odds. Here the likelihood ratio is $0.8/0.2 = 4$: a cough
multiplies the odds of flu by 4, whatever they were before. Prior odds $0.05/0.95 = 0.0526$ become $0.2105$, i.e. a
probability of $0.2105/1.2105 = 0.174$.

In [ ]:
# ── 8.2 Posterior as a function of the prior (likelihoods 0.8 and 0.2) ──
priors = np.linspace(0.001, 0.999, 400)
fig, ax = plt.subplots(figsize=(6.4, 3.5))
ax.plot(priors, posterior(priors, 0.8, 0.2), color="tab:red", label="posterior P(flu | cough)")
ax.plot(priors, priors, "k--", lw=1, label="no evidence: posterior = prior")
for p0 in [0.05, 0.5]:
    ax.plot(p0, posterior(p0, 0.8, 0.2), "o", color="tab:red")
    ax.annotate(f"prior {p0} → {posterior(p0, 0.8, 0.2):.3f}", (p0, posterior(p0, 0.8, 0.2)),
                xytext=(p0 + 0.08, posterior(p0, 0.8, 0.2) - 0.12), fontsize=8, arrowprops=dict(arrowstyle="->"))
ax.set_xlabel("prior P(flu)"); ax.set_ylabel("posterior P(flu | cough)")
ax.set_title("a cough multiplies the odds of flu by 4"); ax.legend(loc="lower right")
plt.tight_layout(); plt.show()
prior_odds = 0.05 / 0.95
print(f"prior odds {prior_odds:.4f} × likelihood ratio 4 = posterior odds {4 * prior_odds:.4f} → probability {4 * prior_odds / (1 + 4 * prior_odds):.4f}")

🔍 **What just happened?** The red curve lies above the diagonal everywhere: a cough is evidence *for* flu, so the
posterior always exceeds the prior. But how much it rises depends on where you start: from 0.05 to 0.174 (the two
marked points), from 0.5 to 0.8. Near 0 and near 1 the curve hugs the diagonal — very rare and very certain conditions
are hard to move with one piece of evidence. The odds calculation reproduces 0.1739.

### 📘 Lecture notes — Bayes' rule

**Setting:** $(\Omega, \mathcal{A}, P)$; $B$ has happened; $P(A \mid B) = P(A \cap B)/P(B)$ ($P(B) > 0$);
$P(A \cap B) = P(A \mid B)P(B) = P(B \mid A)P(A)$.

**Bayes' rule:** $P(A \mid B) = \dfrac{P(B \mid A)P(A)}{P(B)}$ — prior $P(A)$ (belief before the evidence),
likelihood $P(B \mid A)$ (how likely the evidence is under the hypothesis), posterior $P(A \mid B)$ (updated belief),
evidence $P(B)$. **Another form:** $P(A \mid B) = \dfrac{P(B \mid A)P(A)}{P(B \mid A)P(A) + P(B \mid \overline{A})P(\overline{A})}$.

**Inverse probability reasoning:** forward $P(\text{effect} \mid \text{cause})$, e.g. $P(\text{cough} \mid \text{flu})$;
backward $P(\text{cause} \mid \text{effect})$, e.g. $P(\text{flu} \mid \text{cough})$.

*Worked example (Bob):* prior $P(A) = 0.05$, $P(B \mid A) = 0.8$, $P(B \mid \overline{A}) = 0.2$:
$P(A \mid B) = \frac{0.04}{0.04 + 0.19} \approx 0.174$. With prior $P(A) = 0.5$ (flu much more likely to begin with):
$\frac{0.4}{0.4 + 0.1} = 0.8$.

⚠️ The prior is $P(\text{flu})$, the probability of the hypothesis before the evidence — not the background rate of
the symptom.

> 🎓 **Exam tip.** Label the four quantities before computing, write the denominator with the law of total
> probability, and sanity-check: the posterior must lie between 0 and 1 and move in the direction the evidence points.

✏️ **Your turn.** (1) Bob also has a fever. Suppose $P(\text{fever} \mid \text{flu}) = 0.7$ and $P(\text{fever} \mid
\text{no flu}) = 0.05$, and fever and cough are conditionally independent given flu status. Starting from the
posterior 0.174, update again with the fever. (2) Using the odds form, find the prior for which $P(\text{flu} \mid
\text{cough}) = 0.5$.

---
## 9 · Diagnostic tests and genetic risk

> 📍 **Where we are.** Bayes' rule is the mathematics of every diagnostic and genetic test. Here we meet the
> vocabulary of test performance and the most important practical lesson of conditional probability: rare
> conditions produce mostly false alarms.

A test for a condition $D$ is described by three numbers:

* **prevalence** $P(D)$ — the fraction of the tested population that has the condition (the prior);
* **sensitivity** $P(+ \mid D)$ — the fraction of affected people who test positive;
* **specificity** $P(- \mid \overline{D})$ — the fraction of unaffected people who test negative (so the false-positive
  rate is $1 - $ specificity).

The patient needs the **positive predictive value** $\text{PPV} = P(D \mid +)$ and the **negative predictive value**
$\text{NPV} = P(\overline{D} \mid -)$ — both backward probabilities, given by Bayes' rule:

$$P(D \mid +) = \frac{\text{sens}\cdot\text{prev}}{\text{sens}\cdot\text{prev} + (1 - \text{spec})(1 - \text{prev})}.$$

*In words:* among all positive results, the share that come from truly affected people. Below: a screening test for a
rare condition (illustrative numbers): prevalence 0.5 %, sensitivity 99 %, specificity 97 %.

In [ ]:
# ── 9.1 A screening test: Bayes' rule and a natural-frequency tree (ILLUSTRATIVE numbers) ──
prev, sens, spec = 0.005, 0.99, 0.97
ppv = posterior(prev, sens, 1 - spec)
npv = spec * (1 - prev) / (spec * (1 - prev) + (1 - sens) * prev)
print(f"PPV = P(D | +) = {sens}·{prev} / ({sens}·{prev} + {1 - spec:.2f}·{1 - prev}) = {ppv:.4f}")
print(f"NPV = P(no D | −) = {npv:.6f}")

N = 100_000                                                        # natural frequencies: think of 100,000 people
aff = N * prev; unaff = N - aff
tp, fn, fp, tn = aff * sens, aff * (1 - sens), unaff * (1 - spec), unaff * spec
fig, ax = plt.subplots(figsize=(8.5, 3.6))
boxes = {"root": (0.0, 0.5, f"{N:,} people"),
         "aff": (1.0, 0.8, f"{aff:,.0f} affected"), "unaff": (1.0, 0.2, f"{unaff:,.0f} unaffected"),
         "tp": (2.0, 0.95, f"{tp:,.0f} test +"), "fn": (2.0, 0.65, f"{fn:,.0f} test −"),
         "fp": (2.0, 0.35, f"{fp:,.0f} test +"), "tn": (2.0, 0.05, f"{tn:,.0f} test −")}
for parent, child, label in [("root", "aff", "prevalence 0.5 %"), ("root", "unaff", ""), ("aff", "tp", "sensitivity 99 %"),
                             ("aff", "fn", ""), ("unaff", "fp", "1 − specificity = 3 %"), ("unaff", "tn", "")]:
    (x0, y0, _), (x1, y1, _) = boxes[parent], boxes[child]
    ax.annotate("", xy=(x1 - 0.18, y1), xytext=(x0 + 0.2, y0), arrowprops=dict(arrowstyle="->", color="0.4"))
    if label:
        ax.text((x0 + x1) / 2, (y0 + y1) / 2 + 0.07, label, fontsize=8, ha="center", color="0.3",
                bbox=dict(fc="white", ec="none", pad=0.5))
for key, (x, y, text) in boxes.items():
    colour = "#f4cccc" if key in ("tp", "fp") else "#e8eef7"
    ax.text(x, y, text, ha="center", va="center", fontsize=9, bbox=dict(boxstyle="round", fc=colour, ec="0.5"))
ax.text(2.55, 0.5, f"P(affected | +)\n= {tp:,.0f} / ({tp:,.0f} + {fp:,.0f})\n= {tp / (tp + fp):.3f}", fontsize=10,
        va="center", bbox=dict(boxstyle="round", fc="white", ec="tab:red"))
ax.set_xlim(-0.4, 3.3); ax.set_ylim(-0.1, 1.1); ax.axis("off")
ax.set_title("natural frequencies: most positive results are false alarms when the condition is rare")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Bayes' rule gives $\text{PPV} = 0.142$: a positive result means only about a **1 in 7**
chance of having the condition, despite "99 % sensitivity", while a negative result is extremely reassuring
($\text{NPV} = 0.99995$). The tree tells the same story in counts, which most people find far easier: of 100,000 people,
500 are affected and 495 of them test positive; of the 99,500 unaffected, 3 % — **2,985** — also test positive. So
positives are $495 + 2{,}985 = 3{,}480$, of which only 495 (14.2 %) are real. Ignoring the low prevalence and reading
"99 %" as the answer is the **base-rate fallacy**.

In [ ]:
# ── 9.2 PPV depends on the population tested ──
prevalences = np.logspace(-4, -0.3, 300)
fig, ax = plt.subplots(figsize=(6.4, 3.4))
for sp, colour in [(0.97, "tab:blue"), (0.995, "tab:orange"), (0.9999, "tab:green")]:
    ax.plot(prevalences, posterior(prevalences, 0.99, 1 - sp), color=colour, label=f"specificity {sp}")
ax.axvline(prev, color="0.5", ls=":", lw=1)
ax.set_xscale("log"); ax.set_xlabel("prevalence (log scale)"); ax.set_ylabel("PPV = P(D | +)")
ax.set_title("the same test means different things in different populations (sensitivity 0.99)")
ax.legend(); plt.tight_layout(); plt.show()
for p in [0.0001, 0.005, 0.1]:
    print(f"prevalence {p:<6}: PPV = {posterior(p, 0.99, 0.03):.3f} (spec 0.97)   {posterior(p, 0.99, 0.0001):.3f} (spec 0.9999)")

🔍 **What just happened?** For the same test (sensitivity 0.99, specificity 0.97) the PPV is only 0.003 in a
population where 1 in 10,000 is affected, 0.142 at the 0.5 % prevalence above, and 0.786 in a high-risk group with 10 %
prevalence. Raising the specificity to 0.9999 lifts the PPV at 0.5 % prevalence to 0.980. This is why screening tests
for rare conditions (newborn screening, cancer screening) are followed by a more specific confirmatory test, and why a
test validated in a specialist clinic can mislead in the general population.

### 9.1 Bayesian inference for genetic risk assessment

Suppose we want the probability that a woman develops breast cancer given that she carries a pathogenic *BRCA1*
mutation:

$$P(\text{BC} \mid \text{BRCA1+}) = \frac{P(\text{BRCA1+} \mid \text{BC})\cdot P(\text{BC})}{P(\text{BRCA1+})}.$$

*In words:* the **prior** is the baseline population risk of breast cancer; the **likelihood** is how frequently the
*BRCA1* mutation occurs among breast-cancer patients; the **evidence** is how frequent the mutation is in the whole
population; the **posterior** is the updated risk given the test result. This is the mathematical foundation for
clinical genetic testing and personalised medicine. The numbers below are **illustrative**, chosen to be of a realistic
order of magnitude (lifetime risks for carriers are reported at very roughly 60–70 %); real counselling uses
population- and age-specific estimates.

In [ ]:
# ── 9.3 BRCA1 risk by Bayes' rule (ILLUSTRATIVE numbers) ──
p_bc = 0.12                    # prior: lifetime risk of breast cancer in the population
p_brca_given_bc = 0.0135       # likelihood: fraction of breast-cancer patients who carry a BRCA1 mutation
p_brca = 0.0025                # evidence: carrier frequency in the population (about 1 in 400)
post = p_brca_given_bc * p_bc / p_brca
print(f"P(BC | BRCA1+) = {p_brca_given_bc} × {p_bc} / {p_brca} = {post:.3f}   (prior {p_bc})")
p_brca_given_no_bc = (p_brca - p_brca_given_bc * p_bc) / (1 - p_bc)        # law of total probability, solved
print(f"consistency: P(BRCA1+ | no BC) must be {p_brca_given_no_bc:.5f} so that "
      f"{p_brca_given_bc}·{p_bc} + {p_brca_given_no_bc:.5f}·{1 - p_bc:.2f} = {p_brca_given_bc * p_bc + p_brca_given_no_bc * (1 - p_bc):.4f}")
risk_non_carrier = (1 - p_brca_given_bc) * p_bc / (1 - p_brca)            # Bayes' rule for the complement event
print(f"P(BC | BRCA1−) = {1 - p_brca_given_bc:.4f} × {p_bc} / {1 - p_brca:.4f} = {risk_non_carrier:.4f};  "
      f"relative risk carriers vs non-carriers: {post / risk_non_carrier:.1f}×")

🔍 **What just happened?** A prior risk of 12 % becomes a posterior risk of about **65 %** for a carrier: the mutation is
rare in the population (0.25 %) but about 5 times more frequent among patients (1.35 %), and Bayes' rule converts that
ratio into a large change in risk. The second line checks that the three numbers are mutually consistent: the law of
total probability requires $P(\text{BRCA1+} \mid \text{no BC}) = 0.001$. The last line applies Bayes' rule to the
complementary evidence: a non-carrier's risk is $P(\text{BC} \mid \text{BRCA1−}) = 0.9865 \times 0.12 / 0.9975 =
0.1187$, barely below the population's 0.12 (carriers are too rare to move the average), so a carrier's risk is about
**5.5 times** a non-carrier's. ⚠️ Illustrative numbers only — but the *structure* of the calculation is exactly the one used in
risk-prediction tools.

### 🌉 Bridge — genetic counselling: the healthy sister of a cystic-fibrosis patient

A woman's brother has cystic fibrosis, a recessive disease ($cc$); she is healthy. Both parents must be carriers
($Nc$), so *before* we know anything else she is $NN$, $Nc$ or $cc$ with probabilities $\tfrac14, \tfrac12, \tfrac14$
(Module 16). The evidence "healthy" has likelihood 1 for $NN$ and $Nc$ and 0 for $cc$. Bayes' rule over the three
hypotheses:

$$P(Nc \mid \text{healthy}) = \frac{1 \cdot \tfrac12}{1 \cdot \tfrac14 + 1 \cdot \tfrac12 + 0 \cdot \tfrac14} = \frac23.$$

*In words:* being healthy rules out $cc$ but not the carrier state, so her carrier risk *rises* from 1/2 to 2/3. The same
Bayesian bookkeeping — prior from Mendel, likelihoods from phenotypes and test results, posterior for counselling — is
worked through in [BiologyToML 02 · Probability for biologists](../../../../ml/tutorial/BiologyToML/notebooks/02_probability_for_biologists.ipynb).

In [ ]:
# ── 9.4 Bayes over three genotypes: the healthy sister ──
prior_g = {"NN": Fraction(1, 4), "Nc": Fraction(1, 2), "cc": Fraction(1, 4)}
lik_healthy = {"NN": 1, "Nc": 1, "cc": 0}
evidence = sum(prior_g[g] * lik_healthy[g] for g in prior_g)
post_g = {g: prior_g[g] * lik_healthy[g] / evidence for g in prior_g}
print("evidence P(healthy) =", evidence, "  posterior:", {g: str(p) for g, p in post_g.items()})

🔍 **What just happened?** The evidence $P(\text{healthy}) = 3/4$, and the posterior is $P(NN) = 1/3$, $P(Nc) = 2/3$,
$P(cc) = 0$. Bayes' rule works for any number of hypotheses: multiply each prior by its likelihood and divide by the sum
of those products (the law of total probability over the partition of hypotheses).

### 📘 Lecture notes — real-world applications of Bayes' rule

**Diagnostic tests:** prevalence $P(D)$ (prior), sensitivity $P(+ \mid D)$, specificity $P(- \mid \overline{D})$;
$\text{PPV} = P(D \mid +)$ by Bayes' rule. Rare condition ⇒ most positives are false (base-rate fallacy); natural
frequencies make it transparent. *Example:* prevalence 0.005, sensitivity 0.99, specificity 0.97 ⇒ PPV ≈ 0.142.

**Bayesian inference for genetic risk assessment:**
$P(\text{BreastCancer} \mid \text{BRCA1+}) = \dfrac{P(\text{BRCA1+} \mid \text{BreastCancer})\,P(\text{BreastCancer})}{P(\text{BRCA1+})}$
— prior: baseline population risk of breast cancer; likelihood: how frequently the *BRCA1* mutation occurs among
cancer patients; posterior: updated risk given the test result. The mathematical foundation of clinical genetic testing
and personalised medicine.

> 🎓 **Exam tip.** In a test question, write the 2 × 2 table of counts for 100,000 (or 10,000) people first; every
> requested probability is then a ratio of two counts.

✏️ **Your turn.** (1) A second, independent test (sensitivity 0.95, specificity 0.999) is given to everyone who tested
positive in cell 9.1. What is $P(D \mid +, +)$? (Use the PPV 0.142 as the new prior.) (2) In the sister example, she
takes a carrier test with sensitivity 0.9 and specificity 1 and tests **negative**. Update her carrier probability.

---
## 10 · Naïve Bayes: classifying with Bayes' rule and conditional independence

> 📍 **Where we are.** Bayes' rule computes the probability of a hypothesis from one piece of evidence. A
> **classifier** must weigh many pieces at once — many words of an e-mail, many genes of a cell. Combining Bayes' rule
> with a conditional-independence assumption gives one of the simplest and most robust classifiers.

We want the class $C$ (spam or not; T cell or B cell …) of a data point with features $x_1, \ldots, x_n$. Bayes' rule
and the chain rule give

$$P(C \mid x_1, \ldots, x_n) = \frac{P(x_1, \ldots, x_n \mid C)\,P(C)}{P(x_1, \ldots, x_n)}.$$

*In words:* the posterior probability of a class is the likelihood of all the features under that class, times the
prior of the class, divided by the evidence. The **naïve-Bayes assumption** is that the features are **conditionally
independent given the class**, so the joint likelihood factorises (Section 5):

$$P(C \mid x_1, \ldots, x_n) \;\propto\; P(C)\prod_{i=1}^{n} P(x_i \mid C).$$

*In words:* score each class by its prior times the product of the per-feature likelihoods. The symbol $\propto$
("proportional to") means we dropped the denominator $P(x_1, \ldots, x_n)$, which is the same for every class: to
*rank* classes we do not need it, and to get probabilities we divide the scores by their sum.

🧮 **Math refresher — Π.** $\prod_{i=1}^n a_i = a_1 \times a_2 \times \cdots \times a_n$, the product analogue of
$\sum$ (Module 00).

**Workflow.**
1. Estimate $P(C)$ from the class frequencies in the training data.
2. Estimate $P(x_i \mid C)$ for every feature from the training data of each class.
3. Predict the class with the highest posterior.

**Example: spam classification.** Classes: spam ($S$) vs not spam ($\neg S$); features: words such as "free", "click",
"offer". Given $P(S) = 0.7$, $P(\text{"free"} \mid S) = 0.9$ and $P(\text{"free"} \mid \neg S) = 0.01$, for a text that
contains "free":

In [ ]:
# ── 10.1 The spam example ──
p_s, p_free_s, p_free_not_s = 0.7, 0.9, 0.01
score_s = p_free_s * p_s                         # P("free" | S) P(S)
score_not_s = p_free_not_s * (1 - p_s)           # P("free" | ¬S) P(¬S)
print(f"P(S | 'free')  ∝ {p_free_s} × {p_s} = {score_s:.3f}")
print(f"P(¬S | 'free') ∝ {p_free_not_s} × {1 - p_s:.1f} = {score_not_s:.3f}")
print(f"normalised: P(S | 'free') = {score_s:.3f} / ({score_s:.3f} + {score_not_s:.3f}) = {score_s / (score_s + score_not_s):.4f}")

🔍 **What just happened?** The unnormalised scores are $0.9 \times 0.7 = 0.63$ for spam and $0.01 \times 0.3 = 0.003$
for not-spam — ⚠️ note $0.003$, not $0.03$: a slip of one decimal place changes the normalised answer from 0.995 to
0.955. Spam wins, and dividing by the total score gives the posterior $P(S \mid \text{"free"}) = 0.63/0.633 = 0.9953$:
the text is very likely spam.

### 10.1 Naïve Bayes for single-cell RNA-seq classification

In single-cell analysis we often want to classify a cell by its gene expression (neurons vs astrocytes in a brain, or
the immune and stromal cells of our course's simulated tissue):

$$P(\text{CellType} \mid \text{Gene}_1, \ldots, \text{Gene}_n) \;\propto\; P(\text{CellType}) \prod_{i=1}^{n} P(\text{Gene}_i \mid \text{CellType}).$$

*In words:* the assumption is that, **within a cell type**, the expression levels of different genes are
conditionally independent. Each feature is a **count** — the number of mRNA molecules of gene $i$ detected in the
cell — so we need $P(\text{count} = k \mid \text{type})$. A standard model for counts of rare, independent events is
the **Poisson distribution** with mean $\lambda$ ([Module 18 · 🎰 Random variables and their distributions](18_random_variables_distributions.ipynb) derives it):

$$P(X = k) = \frac{\lambda^k e^{-\lambda}}{k!}, \qquad k = 0, 1, 2, \ldots$$

*In words:* if a gene produces on average $\lambda$ detected molecules per cell of a given type, this formula gives
the probability of seeing exactly $k$ of them. Each cell type $t$ gets its own mean $\lambda_{t,i}$ for every gene $i$,
estimated as the average count in the training cells of that type (the maximum-likelihood estimate, [Module 23 · 🎯 Parameter estimation and maximum likelihood](23_maximum_likelihood.ipynb)).

**Avoid underflow: work with logarithms.** A product of 40 small probabilities is tiny, and with the thousands of genes
of a real experiment it falls below the smallest positive floating-point number ($\approx 10^{-308}$) and becomes 0.
Because $\log$ turns products into sums, we score classes by

$$\log P(t) + \sum_{i=1}^{n}\big(x_i \log \lambda_{t,i} - \lambda_{t,i} - \log x_i!\big),$$

*In words:* the log-prior plus, for each gene, the logarithm of its Poisson probability. The class with the largest
log-score also has the largest posterior, because $\log$ is increasing.

In [ ]:
# ── 10.2 SIMULATED single-cell counts, a stratified train/test split and a Poisson naïve Bayes ──
cells_data = simulate_cell_counts()
X, labels, gene_names = cells_data["counts"], cells_data["labels"], cells_data["genes"]
type_names, type_sizes = np.unique(labels, return_counts=True)
print(cells_data["source"], "—", X.shape[0], "cells ×", X.shape[1], "genes;",
      {str(t): int(c) for t, c in zip(type_names, type_sizes)})

rng = np.random.default_rng(SEED + 102)
is_train = np.zeros(len(labels), dtype=bool)
for t in np.unique(labels):                                     # half of every cell type for training
    members = np.flatnonzero(labels == t)
    is_train[rng.choice(members, size=len(members) // 2, replace=False)] = True


def fit_poisson_nb(X_train: np.ndarray, y_train: np.ndarray, pseudocount: float = 0.5):
    """Class priors and per-class Poisson means λ (average count; a small pseudocount keeps λ > 0)."""
    types = np.unique(y_train)
    log_prior = np.log([np.mean(y_train == t) for t in types])
    lam = np.array([(X_train[y_train == t].sum(axis=0) + pseudocount) / ((y_train == t).sum() + 1) for t in types])
    return types, log_prior, lam


def poisson_nb_log_scores(X_new: np.ndarray, log_prior: np.ndarray, lam: np.ndarray) -> np.ndarray:
    """log P(type) + Σ_i log Poisson(x_i; λ_type,i) for every cell (rows) and type (columns)."""
    return log_prior + X_new @ np.log(lam).T - lam.sum(axis=1) - gammaln(X_new + 1).sum(axis=1, keepdims=True)


types, log_prior, lam = fit_poisson_nb(X[is_train], labels[is_train])
print("estimated priors:", {str(t): round(float(p), 3) for t, p in zip(types, np.exp(log_prior))})
print(pd.DataFrame(lam[:, :12], index=types, columns=gene_names[:12]).round(1).to_string())
scores = poisson_nb_log_scores(X[~is_train], log_prior, lam)
check = log_prior[0] + stats.poisson.logpmf(X[~is_train][0], lam[0]).sum()
print(f"\nour log-score for the first test cell / type '{types[0]}': {scores[0, 0]:.4f}  (scipy.stats.poisson: {check:.4f})")

🔍 **What just happened?** `simulate_cell_counts` produced 440 cells (160 T cells, 120 B cells, 90 macrophages, 70
fibroblasts) × 40 genes: 12 marker genes (`CELL_TYPE_MARKERS`) and 28 background genes. We trained on half of each type
(220 cells). The estimated priors are the training-set class frequencies (B cell 0.273, T cell 0.364, fibroblast
0.159, macrophage 0.205). The table of estimated means $\lambda$ shows the marker structure: *CD3E*, *CD3D*, *IL7R*
average 6.5, 9.1 and 9.7 molecules in T cells and at most 0.5 elsewhere; each type's three markers stand out in the
same way. Our hand-written log-score agrees with SciPy's Poisson
log-probabilities to every printed digit. The tiny pseudocount (added to the counts before averaging) keeps every
$\lambda > 0$, so that $\log \lambda$ is defined even for a gene never seen in a type — the idea of pseudocounts returns
in [Module 24 · 🧭 Bayesian estimation and maximum a posteriori](24_bayesian_estimation_map.ipynb).

In [ ]:
# ── 10.3 Posteriors, accuracy and the confusion table on the held-out half ──
def softmax_rows(log_scores: np.ndarray) -> np.ndarray:
    """Turn log-scores into posterior probabilities: subtract the row maximum (avoids underflow), exponentiate, normalise."""
    shifted = log_scores - log_scores.max(axis=1, keepdims=True)
    w = np.exp(shifted)
    return w / w.sum(axis=1, keepdims=True)


post_cells = softmax_rows(scores)
predicted = types[post_cells.argmax(axis=1)]
truth = labels[~is_train]
print(f"accuracy on {len(truth)} held-out cells: {np.mean(predicted == truth):.3f}")
print(pd.crosstab(pd.Series(truth, name="true type"), pd.Series(predicted, name="predicted")))
example = pd.DataFrame(post_cells[:3], columns=types, index=[f"test cell {i} ({truth[i]})" for i in range(3)])
print("\nposterior probabilities of three test cells:\n", example.map(lambda v: f"{v:.2e}").to_string())

for name, cols in [("markers only (12 genes)", slice(0, 12)), ("background only (28 genes)", slice(12, 40))]:
    t_, lp_, lam_ = fit_poisson_nb(X[is_train][:, cols], labels[is_train])
    pred_ = t_[poisson_nb_log_scores(X[~is_train][:, cols], lp_, lam_).argmax(axis=1)]
    print(f"accuracy with {name}: {np.mean(pred_ == truth):.3f}")

🔍 **What just happened?** With all 40 genes the classifier labels every held-out cell correctly (accuracy 1.000, a
diagonal confusion table), and the posteriors are essentially 0 or 1 — the simulated markers are very strong. The
markers alone suffice; the 28 background genes alone give an accuracy of only 0.491 (they differ between types only by
up to ±25 %; always guessing "T cell" would already score 0.364), so almost all of the information sits in a few
marker genes. Real single-cell data are noisier
(over-dispersed counts, drop-outs, intermediate states) and real classifiers are rarely perfect — but the structure
of the computation is the same.

Finally, why the logarithms? The product of the per-gene probabilities for one cell is small but representable here;
with the 2,000+ genes of a real experiment it is not.

In [ ]:
# ── 10.4 Underflow: products of many probabilities vs sums of logarithms ──
cell = X[~is_train][0]
per_gene = stats.poisson.pmf(cell, lam[list(types).index(truth[0])])        # P(x_i | true type) for 40 genes
print(f"40 genes:   product = {np.prod(per_gene):.3e}    sum of logs = {np.sum(np.log(per_gene)):.2f}")
many = np.tile(per_gene, 50)                                                 # pretend the cell had 2,000 such genes
print(f"2,000 genes: product = {np.prod(many):.3e}  ← underflow to 0    sum of logs = {np.sum(np.log(many)):.1f}")
print("smallest positive float64:", np.finfo(float).tiny, "(normal) /", 5e-324, "(subnormal)")

🔍 **What just happened?** For one test cell the 40 per-gene probabilities multiply to $1.2 \times 10^{-24}$ — still a
valid float — and the sum of their logarithms is $-55.1$. Repeating the genes 50
times (as if the cell had 2,000 genes) drives the product below the smallest positive floating-point number, and it
**underflows to exactly 0**: every class would score 0 and the classifier could not decide. The sum of logs is simply
50 times larger ($-2{,}756$) and perfectly usable. Every serious probabilistic program works with log-probabilities, and
`softmax_rows` converts them back safely by subtracting the maximum before exponentiating.

### 📘 Lecture notes — the naïve Bayes classifier

**Idea:** use Bayes' theorem to classify data points from their features, assuming the features are conditionally
independent given the class:

$$P(C \mid x_1, \ldots, x_n) \propto P(C)\prod_{i=1}^{n} P(x_i \mid C).$$

*In words:* prior of the class times the product of per-feature likelihoods; the evidence is common to all classes and
can be dropped for ranking.

**Workflow:** (1) estimate $P(C)$ from the class frequencies; (2) estimate $P(x_i \mid C)$ from the training data;
(3) predict the class with the highest posterior.

*Spam example:* $P(S) = 0.7$, $P(\text{"free"} \mid S) = 0.9$, $P(\text{"free"} \mid \neg S) = 0.01$:
$P(S \mid \text{"free"}) \propto 0.9 \times 0.7 = 0.63$; $P(\neg S \mid \text{"free"}) \propto 0.01 \times (1 - 0.7) =
0.003$ ⇒ more likely spam; normalised posterior $0.63/0.633 \approx 0.995$.

*Single-cell RNA-seq:* $P(\text{CellType} \mid \text{Gene}_1, \ldots, \text{Gene}_n) \propto P(\text{CellType})
\prod_i P(\text{Gene}_i \mid \text{CellType})$, assuming gene expressions are conditionally independent given the cell
type (e.g. classifying cells into neurons, astrocytes, …). Use log-probabilities to avoid underflow.

> 🎓 **Exam tip.** Show where the conditional-independence assumption enters (the step from $P(x_1, \ldots, x_n \mid C)$
> to $\prod_i P(x_i \mid C)$), and keep the unnormalised scores separate from the normalised posteriors.

### 🌉 Bridge — naïve Bayes and automatic cell-type annotation

Assigning cell types is the first step of nearly every single-cell study ([BiologyToML 30 · Single-cell RNA-seq](../../../../ml/tutorial/BiologyToML/notebooks/30_single_cell_rnaseq.ipynb)), and many annotation tools are,
at heart, likelihood-based classifiers: they learn a per-type expression profile from a labelled reference atlas and
assign each new cell to the type under which its counts are most probable — exactly cells 10.2–10.3, usually with
over-dispersed (negative-binomial) instead of Poisson counts ([BiologyToML 24 · GLMs for count data](../../../../ml/tutorial/BiologyToML/notebooks/24_glm_count_data.ipynb)). The naïve assumption is clearly false (genes in
a pathway are co-regulated), yet the *ranking* of classes is often right — a recurring theme of [BiologyToML 08 · k-nearest neighbours and naïve Bayes](../../../../ml/tutorial/BiologyToML/notebooks/08_knn_naive_bayes.ipynb), which also shows
how naïve Bayes compares with k-nearest neighbours and how to estimate its parameters by counting.

✏️ **Your turn.** (1) Simulate noisier data with `simulate_cell_counts(overdispersion=1.0)` and re-run cells 10.2–10.3.
What happens to the accuracy and to the confusion table? (2) Use only **one** marker gene per type (e.g. columns 0, 3, 6,
9). Which pairs of types get confused? (3) Add a second word to the spam example: $P(\text{"click"} \mid S) = 0.4$,
$P(\text{"click"} \mid \neg S) = 0.05$. What is $P(S \mid \text{"free"}, \text{"click"})$ under the naïve-Bayes
assumption?

## 📌 Key takeaways

- **Conditioning restricts the sample space:** $P(A \mid B) = P(A \cap B)/P(B)$ for $P(B) > 0$; in data it means
  "count within the subgroup where $B$ happened". $P(A \mid \Omega) = P(A)$.
- **Sum rule / law of total probability:** $P(B) = \sum_i P(B \cap A_i) = \sum_i P(B \mid A_i)P(A_i)$ over a partition
  — how margins of a joint table are computed (flu/cough: $P(C) = 0.45$, $P(F \mid C) = 2/3$).
- **Product rule** $P(A \cap B) = P(A \mid B)P(B)$ and **chain rule** $P(D_1 \cap \cdots \cap D_n) = P(D_1)P(D_2 \mid
  D_1)\cdots$ hold for any events — no independence needed.
- **Independence** $P(A \cap B) = P(A)P(B)$ ⇔ $P(A \mid B) = P(A)$ (if $P(B) > 0$); disjoint ≠ independent.
  **Conditional independence** $P(A \cap B \mid C) = P(A \mid C)P(B \mid C)$; neither implies the other (coin box: CI
  but dependent; die: independent but not CI given $C$).
- DNA is full of dependence: CpG depletion, codon structure (stop codons: positions 2–3 dependent), motifs and
  **linkage disequilibrium**.
- $P(\cdot \mid B)$ is a probability measure (three axioms), but $P(A \mid B \cup C) \ne P(A \mid B) + P(A \mid C)$
  and $P(A \mid B) \ne P(B \mid A)$.
- **Bayes' rule:** posterior = likelihood × prior / evidence; the prior matters (Bob: 0.174 vs 0.8); rare conditions make
  most positive tests false alarms (base-rate fallacy; natural frequencies).
- **Naïve Bayes:** $P(C \mid \mathbf{x}) \propto P(C)\prod_i P(x_i \mid C)$; compute with log-probabilities; a Poisson
  naïve Bayes types simulated single cells from their counts.

---
## ✅ Checkpoint

1. Two fair dice are rolled. What is $P(\text{sum} = 8 \mid \text{first die} = 3)$? And $P(\text{first die} = 3 \mid \text{sum} = 8)$?
   <details><summary>Answer</summary>Given first = 3 the sum is 8 only if the second is 5: probability 1/6. The event "sum = 8" has 5 outcomes (2,6), (3,5), (4,4), (5,3), (6,2); one of them has first = 3, so P(first = 3 | sum = 8) = 1/5.</details>
2. From the flu/cough table compute $P(C \mid F)$ and $P(\overline{F} \mid \overline{C})$.
   <details><summary>Answer</summary>P(C | F) = 0.30/0.50 = 0.6. P(C̄) = 0.20 + 0.35 = 0.55, so P(F̄ | C̄) = 0.35/0.55 ≈ 0.636.</details>
3. Use the chain rule: a cage holds 6 mice, 2 of them knockouts. You take out 2 mice one after the other. What is the probability that both are knockouts?
   <details><summary>Answer</summary>P(D₁) P(D₂ | D₁) = (2/6)(1/5) = 1/15.</details>
4. Prove that if $P(A \mid B) = P(A)$ (with $P(B) > 0$) then $A$ and $B$ are independent.
   <details><summary>Answer</summary>By the product rule P(A ∩ B) = P(A | B) P(B) = P(A) P(B), which is the definition of independence.</details>
5. Can two disjoint events with $P(A) = 0.3$ and $P(B) = 0.4$ be independent?
   <details><summary>Answer</summary>No: P(A ∩ B) = P(∅) = 0 but P(A)P(B) = 0.12 ≠ 0. Disjoint events with positive probabilities are always dependent.</details>
6. In the coin box (regular + two-headed coin), compute $P(C \mid A)$, the probability that you hold the regular coin after seeing one head.
   <details><summary>Answer</summary>P(C | A) = P(A | C)P(C)/P(A) = (1/2 · 1/2)/(3/4) = 1/3.</details>
7. Give an example of two events that are independent but not conditionally independent given a third event.
   <details><summary>Answer</summary>The die: A = {1, 2}, B = {2, 4, 6} are independent (1/6 = 1/3 · 1/2), but given C = {1, 4}, P(A ∩ B | C) = 0 ≠ 1/4 = P(A | C) P(B | C).</details>
8. In the stop codons TAG, TAA, TGA, compute $P(\text{2nd} = G \mid \text{3rd} = A)$ and $P(\text{2nd} = G)$. Are positions 2 and 3 independent?
   <details><summary>Answer</summary>3rd = A in TAA and TGA, of which one has 2nd = G: P = 1/2. But P(2nd = G) = 1/3. They differ, so the positions are dependent.</details>
9. A test has sensitivity 0.9 and specificity 0.95; prevalence is 2 %. Compute $P(D \mid +)$.
   <details><summary>Answer</summary>P(+) = 0.9·0.02 + 0.05·0.98 = 0.018 + 0.049 = 0.067; P(D | +) = 0.018/0.067 ≈ 0.269.</details>
10. Bob's flu (likelihoods 0.8 and 0.2): what prior gives a posterior of exactly 0.5?
    <details><summary>Answer</summary>Posterior odds = 4 × prior odds = 1 ⇒ prior odds 1/4 ⇒ prior = 1/5 = 0.2. Check: 0.16/(0.16 + 0.16) = 0.5.</details>
11. In the spam example someone writes $P(\neg S \mid \text{"free"}) \propto 0.01 \times 0.3 = 0.03$. What is the correct value, and how does the error change the normalised posterior?
    <details><summary>Answer</summary>0.01 × 0.3 = 0.003. Correct posterior 0.63/0.633 ≈ 0.995; with the slip it would be 0.63/0.66 ≈ 0.955.</details>
12. Why does a naïve-Bayes program add log-probabilities instead of multiplying probabilities?
    <details><summary>Answer</summary>Products of thousands of probabilities underflow to 0 in floating point; logs turn the product into a sum of moderate numbers, and because log is increasing the class with the largest log-score is the class with the largest posterior.</details>

## 🧠 Questions & discussion

1. A study reports that 70 % of patients with a disease carry variant X. A newspaper writes "carriers of X have a 70 % chance of the disease". Which conditional probability is which, and what extra information would you need?
2. Screening programmes (for example newborn screening) accept many false positives. Why might that be the right decision, and what must follow a positive screen?
3. Naïve Bayes assumes that genes are conditionally independent given cell type. Name a biological reason why this is false, and guess whether it makes the posteriors over- or under-confident.
4. Conditional independence of two tosses *given the coin* turned into dependence once the coin was unknown. Find a biological analogue in which two measurements are independent given a hidden variable (disease state, cell type, batch) but correlated in the pooled data.
5. LD makes nearby SNPs dependent. Is that a problem or an opportunity for a genome-wide association study? Argue both sides.

## 🚀 Bonus — going further

- **Simpson's paradox.** Construct a 2 × 2 × 2 table in which a treatment helps within each of two hospitals but appears to harm in the pooled data. Which conditional probabilities reverse, and why does the law of total probability allow it?
- **The Monty Hall problem.** Simulate it, then solve it with Bayes' rule over the three doors.
- **Multiple tests and correlated errors.** Repeat the double-testing exercise of Section 9 when the two tests' errors are *not* conditionally independent (e.g. both are fooled by the same cross-reacting antibody). How much does the second positive add?
- **Gene finding as Bayes' rule.** Given emission probabilities for coding and non-coding DNA, compute the posterior probability that a 100-bp window is coding — the first step towards the HMMs of [Module 20 · ⛓️ Graphical models: Bayesian networks and Markov chains](20_bayesian_networks_markov_chains.ipynb) and [BiologyToML 19 · Hidden Markov models: inference](../../../../ml/tutorial/BiologyToML/notebooks/19_hmm_inference.ipynb).

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module 17 — Conditional probability, independence and Bayes' rule — date:
Input: two-coin experiments, the flu/cough table, a litter of mice, DNA (independent, background
       Markov, stop codons), haplotypes, a deck of cards, test results, SIMULATED single-cell counts
Output: conditional probabilities (exact and simulated); independence / CI checks; PPV curves;
        BRCA1 posterior; naïve-Bayes accuracy and confusion table
Method: P(A|B) = P(A∩B)/P(B); sum, product, chain rules; Bayes' rule; Poisson naïve Bayes in logs
Key equation: P(A|B) = P(B|A)P(A) / [P(B|A)P(A) + P(B|Ā)P(Ā)]
Key parameters: prior / prevalence, sensitivity, specificity, likelihood ratio; λ per gene and type
What changed when I modified the prior / the specificity / the overdispersion:
One failure mode: (e.g., confusing P(A|B) with P(B|A); assuming independence that does not hold)
Connection to 02-680 (which idea of conditional probability and Bayes' rule this illustrates):
```

## 📚 References

- Wasserman L. (2004) *All of Statistics: A Concise Course in Statistical Inference*. Springer — Chapter 1 (conditional probability, independence, Bayes' theorem).
- DeGroot M. H., Schervish M. J. (2012) *Probability and Statistics*, 4th ed. Pearson — Chapter 2 (conditional probability, independent events, Bayes' theorem).
- Deisenroth M. P., Faisal A. A., Ong C. S. (2020) *Mathematics for Machine Learning*. Cambridge University Press — Sections 6.3 (sum rule, product rule, Bayes' theorem) and 6.4.5 (conditional independence).
- Betti M. *Mathematics and Statistics for the Quantitative Sciences* — the conditional-probability chapter.
- Liben-Nowell D. (2022) *Connecting Discrete Mathematics and Computer Science*. Cambridge University Press — conditional probability and independence.
- D. DeBlasio, 02-680 lecture notes, github.com/deblasiolab/02-680_Documents.
- Bayes T. (1763) An essay towards solving a problem in the doctrine of chances. *Philosophical Transactions of the Royal Society of London* 53:370–418.
- Bird A. P. (1980) DNA methylation and the frequency of CpG in animal DNA. *Nucleic Acids Research* 8:1499–1504.
- Slatkin M. (2008) Linkage disequilibrium — understanding the evolutionary past and mapping the medical future. *Nature Reviews Genetics* 9:477–485.
- Gigerenzer G., Hoffrage U. (1995) How to improve Bayesian reasoning without instruction: frequency formats. *Psychological Review* 102:684–704.

## ➡️ Next

[Module 18 · 🎰 Random variables and their distributions](18_random_variables_distributions.ipynb) — so far we asked yes/no questions about outcomes (is it heads? does she have the flu?). Most biological
measurements are **numbers**: the number of mutant alleles, of reads on a gene, of mutations in a region; a height, a
concentration, a waiting time. A **random variable** is a function that turns each outcome into a number, and its
**distribution** (PMF, PDF, CDF) summarises all the probabilities at once. The next module builds that machinery and the
famous families — Bernoulli, binomial, Poisson (the one we borrowed for naïve Bayes), geometric, uniform, Gaussian and
exponential.